# Spoken Language Processing 2025-26

# Lab3 - Dialogue Systems

_Bruno Martins_


This lab assignment will introduce tools and concepts related to the development of dialogue systems, exemplifying also the use of automatic speech recognition and text-to-speech models in this particular context. The assignment is also associated with the [FIDAWARD IN Spoken Language Processing award](https://tt.tecnico.ulisboa.pt/en/parcerias-empresariais/aproximacao-ao-talento/premios-de-merito-alunos/english-premio-de-merito-fidaward-in-spoken-language-processing-powered-by-fidelidade/).

Students will be tasked with the development of a turn-based spoken/conversational question answering system, reusing different models available from within the catalogue associated to the HuggingFace Transformers library:

* Speech recognition models (e.g., OpenAI Whisper or SpeechT5).
* Large language models for natural language understanding and generation (e.g., the [Qwen3.5](https://huggingface.co/Qwen/Qwen3.5-0.8B) or [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) models).
* Text-to-speech models (e.g., SpeechT5).

The first parts of this notebook will guide students in the use of the tools, while the last part presents the main problem that is to be tackled. Note that the first parts also feature **intermediate tasks which students are required to solve**.

To complete the project, student groups must deliver in Fenix an **updated version of this notebook**, featuring the proposed solutions to each task, together with a **small PDF report (2 pages)** outlining the methods that were developed (use the [following Overleaf template](https://www.overleaf.com/latex/templates/interspeech-2026-paper-kit/kzcdqdmkqvbr) for the report). The report can contain a section for each of the parts in the notebook. The set of files corresponding to the solution to Lab3 should be uploaded in Fenix through a .zip file named after the number of the group.

Students are encouraged to modify examples, incorporate different techniques, and in general explore any approach that may permit improving the results. Assessment will be based on task completion, **creativity in the proposed solutions**, and overall accuracy over a benchmark dataset.

### Group identification

Initialize the variable `group_id` with the number that Fenix assigned to your group and `student1_name`, `student1_id`, `student2_name` and `student2_id` with your names and student numbers.

In [ ]:
# YOUR CODE HERE
group_id=7
student1_id=106261
student1_name = "Dinis Alves da Silva"
student2_id=106362
student2_name = "Henrique Rodrigues"
print(f"Group number: {group_id}")
print(f"Student 1: {student1_name} ({student1_id})")
print(f"Student 2: {student2_name} ({student2_id})")

: 

In [ ]:
assert isinstance(group_id, int) and isinstance(student1_id, int) and isinstance(student2_id, int)
assert isinstance(student1_name, str) and isinstance(student2_name, str)
assert (group_id > 0) and (group_id < 40)
assert (student1_id > 60000) and (student1_id < 120000) and (student2_id > 60000) and (student2_id < 120000)

# Install and import Python packages

NumPy is a Python library that provides functions to process multidimensional arrays. The NumPy documentation is available [here](https://numpy.org/doc/1.24/).

[Librosa](https://librosa.org/) is a Python package for analyzing and processing audio signals. It provides a wide range of tools for tasks such as loading and manipulating audio files, extracting features from audio signals, and visualizing and playing back audio data.

IPython display is a module in the IPython interactive computing environment that provides a set of functions for displaying various types of media in the Jupyter notebook or other IPython-compatible environments. For example, you can use the display() function to display an object in a notebook cell (for example an audio object).

Matplotlib is a popular Python library that allows users to create a wide range of visualizations using a simple and intuitive syntax.

Huggingface transformers provides APIs and tools to easily download and train state-of-the-art pretrained models based on the Transformer architecture. The documentation is available [here](https://huggingface.co/docs/transformers/index) and, for more details, you can check the official [HuggingFace course](https://huggingface.co/course/chapter1/1).

Two libraries associated to HuggingFace transformers, named [datasets](https://huggingface.co/docs/datasets/index) and [evaluate](https://huggingface.co/docs/evaluate/index), respectivly suport the direct access to many well-known datasets and common evaluation metrics used in NLP and speech processing research.

In [ ]:
import sys
!{sys.executable} -m pip install -U transformers
!{sys.executable} -m pip install -U fsspec==2025.3.0
!{sys.executable} -m pip install -U jiwer
!{sys.executable} -m pip install -U librosa
!{sys.executable} -m pip install "datasets<3.0.0"
!{sys.executable} -m pip install -U evaluate
!{sys.executable} -m pip install -U bitsandbytes accelerate
!{sys.executable} -m pip install -U peft sentencepiece

In [ ]:
import evaluate
import datasets
import transformers
import numpy as np
import librosa
import librosa.display
from IPython.display import Audio
from matplotlib import pyplot as plt
from transformers import logging

logging.set_verbosity(logging.CRITICAL)

# Using OpenAI Whisper

Whisper is a cutting-edge model for for Automatic Speech Recognition (ASR), developed by OpenAI using a massive dataset of 680,000 hours of multilingual and multitask supervised data collected from the internet, and made available through the HuggingFace Transformers library.

The following example illustrates the use of the Whisper model to transcribe a small audio sample taken from the LibriSpeech dataset (which is available through the HuggingFace datasets library).

More detailed information about Whisper, including information on how to fine-tune the model with task-specific data, is available on a [tutorial in the HuggingFace blog](https://huggingface.co/blog/fine-tune-whisper).

In [ ]:
import torch
import librosa
from transformers import logging
from IPython.display import Audio
from transformers import AutoProcessor
from transformers import WhisperForConditionalGeneration, SpeechT5ForSpeechToText
from datasets import load_dataset

# Use the datasets library in streaming mode to avoid loading the entire dataset during initialization
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation", streaming=True)

processor = AutoProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Instead of using "openai/whisper-small", you can alternatively try the "microsoft/speecht5_asr" ASR model
# processor = AutoProcessor.from_pretrained("microsoft/speecht5_asr")
# model = SpeechT5ForSpeechToText.from_pretrained("microsoft/speecht5_asr")

# Access the dataset as an Iterator and retrieve the first element in the dataset
audio = next(iter(ds))["audio"]["array"]

# Resample audio to 16kHz (not needed in the case of the particular dataset used in this example)
# audio = librosa.resample(audio, orig_sr=16000, target_sr=16000)

inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")

# You are able to hear the audio
display(Audio(audio, rate=16000))

# You can use the option task="translate" to perform speech translation
generated_ids = model.generate(**inputs)
transcription = processor.batch_decode(generated_ids, max_length=250, skip_special_tokens=True)[0]

print(transcription)

Automatic Speech Recognition (ASR) models are frequently evaluated through the Word Error Rate ([WER](https://huggingface.co/learn/audio-course/en/chapter5/evaluation#word-error-rate)).

The WER is derived from the Levenshtein distance, working at the word level and aligning the recognized word sequence with the reference (spoken) word sequence using dynamic string alignment. The metric can then be computed as:

WER = (S + D + I) / N = (S + D + I) / (S + D + C),

where S is the number of substitutions, D is the number of deletions, I is the number of insertions, C is the number of correct words, and N is the number of words in the reference (N=S+D+C). The WER value indicates the average number of errors per reference word. The lower the value, the better the performance of the ASR system, with a WER of 0 being a perfect score.

The example below illustrates the computation of the WER for two paired examples of a generated sentence versus a reference sentence. The score produced as output is the average value accross the two examples.

In [ ]:
from evaluate import load

wer = load("wer")
predictions = ["this is the prediction", "there is an other sample"]
references = ["this is the reference", "there is another one"]
wer_score = wer.compute(predictions=predictions, references=references)

print(wer_score)

## Intermediate tasks:

* Collect two audio samples with your own voice, together with an English transcription of the spoken messages. The following [example shows how to record audio from your microphone within a Python notebook running on Google Colab](https://colab.research.google.com/gist/ricardodeazambuja/03ac98c31e87caf284f7b06286ebf7fd/microphone-to-numpy-array-from-your-browser-in-colab.ipynb#scrollTo=H4rxNhsEpr-c), but you can use any other method to collect the audio samples.
* Use the Whisper ASR model to transcribe/translate the two spoken messages that were collected into English text. Notice that Whisper supports speech translation, and hence you can test the model with audio samples involving speech in different languages.
* Use the transcriptions to compute the word error rate.
* Experiment with the use of different recognition models (e.g., larger Whisper models, or more recent ASR models -- check the [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard)) over a larger set of audio/transcription pairs, and see if the error rate changes.

In [ ]:
import sys
!{sys.executable} -m pip install sounddevice -q

In [ ]:
import io, sys
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from IPython.display import Audio, display

AUDIO_DIR = Path(".")          # same folder as the notebook (Lab_3)
AUDIO_FILES = [AUDIO_DIR / "recording_1.wav", AUDIO_DIR / "recording_2.wav"]
SAMPLE_RATE = 16000

# ── environment detection ───────────────────────────────────────────────────
def _is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

# ── recording backends ──────────────────────────────────────────────────────
def _record_local(duration=5):
    import sounddevice as sd
    print(f"  Recording {duration}s — speak now!")
    audio = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                   channels=1, dtype="float32")
    sd.wait()
    print("  Done.")
    return audio.squeeze()

def _record_colab(duration=5):
    from IPython.display import Javascript
    from google.colab import output
    import base64
    display(Javascript("""
    const sleep = t => new Promise(r => setTimeout(r, t));
    const b2text = b => new Promise(r => {
      const rd = new FileReader();
      rd.onloadend = e => r(e.srcElement.result);
      rd.readAsDataURL(b);
    });
    window._recordAudio = t => new Promise(async r => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const rec = new MediaRecorder(stream);
      const chunks = [];
      rec.ondataavailable = e => chunks.push(e.data);
      rec.start();
      await sleep(t);
      rec.onstop = async () => { r(await b2text(new Blob(chunks))); };
      rec.stop();
    });
    """))
    print(f"  Recording {duration}s — speak now!")
    s = output.eval_js(f"_recordAudio({duration * 1000})")
    raw = base64.b64decode(s.split(",")[1])
    # The browser MediaRecorder yields WebM/Opus, which libsndfile cannot read.
    # Decode via ffmpeg (preinstalled on Colab) to 16 kHz mono WAV, then load.
    import subprocess, tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as _f:
        _f.write(raw); _src = _f.name
    _dst = _src[:-5] + ".wav"
    try:
        subprocess.run(["ffmpeg", "-y", "-i", _src, "-ar", str(SAMPLE_RATE), "-ac", "1", _dst],
                       check=True, capture_output=True)
        audio, sr = sf.read(_dst)
    finally:
        for _p in (_src, _dst):
            if os.path.exists(_p):
                os.remove(_p)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    print("  Done.")
    return audio.astype(np.float32)

def _do_record(duration=5):
    return _record_colab(duration) if _is_colab() else _record_local(duration)

# ── slot management ─────────────────────────────────────────────────────────
def _oldest_slot():
    return min(AUDIO_FILES, key=lambda p: p.stat().st_mtime if p.exists() else float("inf"))

def get_or_record(slot_idx=None, force=False, duration=5):
    """Load a saved recording from disk, or record a new one and save it.

    slot_idx : 0 or 1 — which file to use. None = first empty slot,
               or the oldest file when both already exist.
    force    : re-record even if the file already exists.
    duration : recording length in seconds.
    """
    if slot_idx is None:
        empty = [p for p in AUDIO_FILES if not p.exists()]
        path = empty[0] if empty else _oldest_slot()
    else:
        path = AUDIO_FILES[slot_idx]

    if path.exists() and not force:
        print(f"Loaded existing {path.name}  (set force=True to re-record)")
        audio, _ = sf.read(str(path))
        return audio.astype(np.float32)

    print(f"Recording → {path.name}")
    audio = _do_record(duration)
    sf.write(str(path), audio, SAMPLE_RATE)
    print(f"  Saved to {path}")
    return audio

print("Recording helpers ready.")


In [ ]:
# ── Recording 1 ─────────────────────────────────────────────────────────────
# Set FORCE_RECORD_1 = True to record a new audio (overwrites recording_1.wav)
FORCE_RECORD_1 = False

audio_1 = get_or_record(slot_idx=0, force=FORCE_RECORD_1, duration=5)
display(Audio(audio_1, rate=SAMPLE_RATE))


In [ ]:
# ── Recording 2 ─────────────────────────────────────────────────────────────
# Set FORCE_RECORD_2 = True to record a new audio (overwrites recording_2.wav)
FORCE_RECORD_2 = False

audio_2 = get_or_record(slot_idx=1, force=FORCE_RECORD_2, duration=5)
display(Audio(audio_2, rate=SAMPLE_RATE))


In [ ]:
# ── Reference transcriptions ─────────────────────────────────────────────────
# Fill in exactly what you said in each recording (used to compute WER)
reference_1 = "who is the CEO of openAI?"
reference_2 = "Quem é o fundador da Ford?"

print("References:")
print(f"  1: {reference_1}")
print(f"  2: {reference_2}")

In [ ]:
# ── Whisper transcription + WER ───────────────────────────────────────────────
import torch
from transformers import AutoProcessor, WhisperForConditionalGeneration
from evaluate import load

# Reuse the processor/model already loaded above, or reload if needed
try:
    processor, model
except NameError:
    processor = AutoProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

def transcribe(audio, language="en", task="transcribe", proc=None, mdl=None):
    """Transcribe with an EXPLICIT language to avoid wrong language auto-detection.

    Recording 1 is English; recording 2 is Portuguese (see the reference cell). The
    original 50% WER came from leaving language unset, so Whisper-small mis-handled
    recording 2. Pass task='translate' to render non-English speech as English instead.
    """
    proc = proc or processor
    mdl = mdl or model
    inputs = proc(audio=audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: (v.to(mdl.device, dtype=mdl.dtype) if torch.is_floating_point(v)
                  else v.to(mdl.device)) for k, v in inputs.items()}
    try:
        with torch.no_grad():
            generated_ids = mdl.generate(**inputs, language=language, task=task)
    except (TypeError, ValueError):
        # Older transformers: set the decoder prompt ids explicitly.
        forced = proc.get_decoder_prompt_ids(language=language, task=task)
        with torch.no_grad():
            generated_ids = mdl.generate(**inputs, forced_decoder_ids=forced)
    return proc.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

# Recording 1 = English, Recording 2 = Portuguese
pred_1 = transcribe(audio_1, language="en")
pred_2 = transcribe(audio_2, language="pt")

print("Whisper transcriptions (openai/whisper-small):")
print(f"  Recording 1 [en]: {pred_1}")
print(f"  Recording 2 [pt]: {pred_2}")

wer_metric = load("wer")
wer_1 = wer_metric.compute(predictions=[pred_1.lower()], references=[reference_1.lower()])
wer_2 = wer_metric.compute(predictions=[pred_2.lower()], references=[reference_2.lower()])
wer_all = wer_metric.compute(
    predictions=[pred_1.lower(), pred_2.lower()],
    references=[reference_1.lower(), reference_2.lower()],
)
print(f"\nWER recording 1 (en): {wer_1 * 100:5.1f}%")
print(f"WER recording 2 (pt): {wer_2 * 100:5.1f}%")
print(f"WER overall:          {wer_all * 100:5.1f}%")

In [ ]:
# ── Experiment: compare ASR models (Open ASR Leaderboard) ────────────────────
import gc, psutil
from transformers import AutoProcessor as _CAP, WhisperForConditionalGeneration as _CWM
from evaluate import load as _cl

_wer = _cl("wer")

# Model list with approximate RAM/VRAM needed in float32 (CPU) and float16 (GPU)
ASR_MODEL_SIZES = {
    "openai/whisper-small":          0.5,   # GB
    "openai/whisper-medium":         1.5,
    "openai/whisper-large-v3":       6.0,   # too large for CPU — skip automatically
    "distil-whisper/distil-large-v3": 3.0,
}

# On CPU we keep only models that fit comfortably in available RAM.
# On GPU we allow anything that fits in VRAM (the code uses float16 there).
_using_gpu = torch.cuda.is_available()
if _using_gpu:
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _safe_gb = _vram_gb * 0.6          # leave headroom for activations
    print(f"GPU detected: {torch.cuda.get_device_name(0)}  ({_vram_gb:.1f} GB VRAM)")
else:
    _ram_gb  = psutil.virtual_memory().available / 1e9
    _safe_gb = _ram_gb * 0.5           # leave headroom for OS + other processes
    print(f"No GPU — CPU only.  Available RAM: {_ram_gb:.1f} GB  (safe budget: {_safe_gb:.1f} GB)")

ASR_MODELS = [mid for mid, sz in ASR_MODEL_SIZES.items() if sz <= _safe_gb]
skipped    = [mid for mid, sz in ASR_MODEL_SIZES.items() if sz >  _safe_gb]

if skipped:
    print(f"Skipping (too large for this environment): {skipped}")
print(f"Running: {ASR_MODELS}\n")

_recordings = [("rec1", audio_1, "en", reference_1),
               ("rec2", audio_2, "pt", reference_2)]

print(f"{'Model':<38}{'rec1 WER':>10}{'rec2 WER':>10}{'overall':>10}")
print("─" * 68)
asr_comparison = {}

for mid in ASR_MODELS:
    try:
        # Free everything possible before loading the next model
        gc.collect()
        if _using_gpu:
            torch.cuda.empty_cache()

        _adt = torch.float16 if _using_gpu else torch.float32
        print(f"Loading {mid} ...", end=" ", flush=True)
        cp = _CAP.from_pretrained(mid)
        cm = _CWM.from_pretrained(mid, torch_dtype=_adt,
                                  low_cpu_mem_usage=True)
        cm = cm.to("cuda") if _using_gpu else cm
        cm.eval()
        print("ok")

        preds, refs, per = [], [], []
        for label, aud, lang, ref in _recordings:
            lg = "en" if "distil" in mid else lang
            p = transcribe(aud, language=lg, proc=cp, mdl=cm)
            preds.append(p.lower())
            refs.append(ref.lower())
            w = _wer.compute(predictions=[p.lower()], references=[ref.lower()])
            per.append(w)
            print(f"  {label} ({lg}): '{p}'  WER={w*100:.1f}%")

        ov = _wer.compute(predictions=preds, references=refs)
        asr_comparison[mid] = ov
        short = mid.split("/")[-1]
        print(f"{short:<38}{per[0]*100:>9.1f}%{per[1]*100:>9.1f}%{ov*100:>9.1f}%\n")

    except Exception as e:
        print(f"\n  ! {mid} failed: {str(e)[:80]}")
    finally:
        # Always free the model, even on error
        for _obj in ("cp", "cm"):
            if _obj in dir():
                del globals()[_obj] if _obj in globals() else None
        try:
            del cp
        except NameError:
            pass
        try:
            del cm
        except NameError:
            pass
        gc.collect()
        if _using_gpu:
            torch.cuda.empty_cache()

if asr_comparison:
    best = min(asr_comparison, key=asr_comparison.get)
    print(f"Best overall WER: {best}  ({asr_comparison[best]*100:.1f}%)")


# Using LLMs for conditional language generation

[OpenAI GPT-2](https://openai.com/index/gpt-2-1-5b-release/) is a language model based on the Transformer decoder architecture, trained with large scale data collected from the Web using a simple objective: predict the next word, given all of the previous words within some text. The diversity of the dataset causes this simple goal to contain naturally occurring demonstrations of many tasks across diverse domains. Thus, GPT-2 can be used to address problems like question answering, modeling the task as language generation conditioned in the question (plus other relevant additional context).

The following example illustrates the use of the GPT-2 model through the Huggingface Transformers library. The example is written in a generic form, which can easilly be adapted to to be used with other LLMs, including models trained to follow instructions. The example illustrates the use of a particular languade decoding algorithm (i.e., beam search), as well as some of the typicall preprocessing and postprocessing steps, allowing us to directly input any text and getting an intelligible answer.

In [ ]:
from transformers import pipeline, set_seed
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# make results deterministic
set_seed(42)

# You can also try other models instead of "gpt2", e.g. "Qwen/Qwen3.5-0.8B"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

messages = [{"role": "user", "content": "Can you tell us what is the capital city of the UK?"}]
if tokenizer.chat_template is None: input_text = messages[0]["content"]
else: input_text = tokenizer.apply_chat_template(messages, enable_thinking=False, add_generation_prompt=True, tokenize=False)

inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10, num_beams=5)

print("**Sequence handled by the language model**")
print(tokenizer.decode(outputs[0]))
print()
print("**Output**")
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip())


## Intermediate tasks:

* Adapt the example showing how to use GPT-2 to do question answering over the [TriviaQA dataset](http://nlp.cs.washington.edu/triviaqa/) (you can use a [version](https://huggingface.co/datasets/lucadiliello/triviaqa) of this dataset from a previous shared task, which is available from HuggingFace datasets).
* Evaluate the results obtained with different language models. These can include relatively small models trained to follow instructions, for instance from the [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) or [TinyLlama](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) families, open versions of larger models such as those from [Qwen](https://huggingface.co/Qwen/Qwen3.5-0.8B) or [DeepSeek](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B), or models available through APIs such as those supported in the [IAedu platform](https://chat.iaedu.pt/) (in this case using separate Python libraries that support the API calls).
* Evaluate different strategies for improving the use of language models in the question answering task (e.g., considering different prompting strategies, retrieval-augmented generation, models with support for reasoning, parameter efficient fine-tuning, etc.).
* Compute the error over the first 500 examples from the validation split from the TriviaQA dataset, using the [TER metric](https://github.com/huggingface/evaluate/tree/main/metrics/ter) for comparing the generated answers against the ground truth.

Notice that ground-truth answers in TriviaQA correspond to relaively short phrases, e.g. directly answering questions through entity names. In this intermediate step, you should consider designing a strategy that can **generate/extract short answers**.


### Task 1 — GPT-2 Question Answering on TriviaQA

GPT-2 is a pure language model with no instruction tuning — it cannot be told to "answer briefly". Instead we use a **few-shot prompt**: three example Q/A pairs are prepended to the question, guiding the model to continue in the same format (`Q: ... A: ...`). After generation we extract only the first line after the last `A:`, discarding any continuation.

The TriviaQA dataset (`lucadiliello/triviaqa`) is loaded from HuggingFace and provides the questions and a list of valid answer strings per example. The cell below first inspects the dataset structure, then runs GPT-2 on the first 10 questions as a sanity check.

In [ ]:
# ── Task 1 · Load TriviaQA and inspect structure ─────────────────────────────
from datasets import load_dataset

trivia_ds = load_dataset("lucadiliello/triviaqa", split="validation")

print(f"Total validation examples: {len(trivia_ds)}")
print(f"Columns: {trivia_ds.column_names}")
print()
print("First example:")
ex0 = trivia_ds[0]
for k, v in ex0.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Task 1 · GPT-2 few-shot QA on TriviaQA ───────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(42)

# Reuse GPT-2 already loaded in the demo cell above, or reload
try:
    tokenizer, model
    assert hasattr(model, "generate")
except (NameError, AssertionError):
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")

model.eval()

# Few-shot prompt: 3 examples anchor GPT-2 to the Q/A continuation format.
# NOTE: these exemplars are fixed and are NOT drawn from the eval set (no leakage).
FEW_SHOT = (
    "Q: What is the chemical symbol for gold?\n"
    "A: Au\n\n"
    "Q: Who wrote the play Romeo and Juliet?\n"
    "A: William Shakespeare\n\n"
    "Q: In which country is the Eiffel Tower located?\n"
    "A: France\n\n"
)

def build_prompt(question: str) -> str:
    return FEW_SHOT + f"Q: {question}\nA:"

def extract_short_answer(full_text: str, prompt: str) -> str:
    """Return only the first line generated after 'A:' in the new tokens (GPT-2 helper)."""
    new_text = full_text[len(prompt):]
    # Stop at the first newline or at the next "Q:" — whichever comes first
    for sep in ("\n", "Q:"):
        idx = new_text.find(sep)
        if idx != -1:
            new_text = new_text[:idx]
    return new_text.strip()

def answer_gpt2(question: str) -> str:
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            num_beams=5,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_short_answer(full_text, prompt)

def get_answers(example: dict) -> list:
    """Return a de-duplicated list of valid answer strings from a TriviaQA example.

    Handles both the flat `lucadiliello/triviaqa` schema (answers: list[str]) and the
    HuggingFace `trivia_qa` schema (answer: {value, aliases, normalized_aliases}).
    Returning every alias (not just the first) lets the matcher credit any valid form.
    """
    ans = example.get("answers")
    if ans is None:
        ans = example.get("answer", [])
    if isinstance(ans, str):
        cands = [ans]
    elif isinstance(ans, dict):
        cands = []
        for key in ("value", "normalized_value", "aliases", "normalized_aliases"):
            v = ans.get(key)
            if isinstance(v, str):
                cands.append(v)
            elif isinstance(v, (list, tuple)):
                cands.extend(v)
    else:
        cands = list(ans)
    seen, out = set(), []
    for a in cands:
        a = str(a).strip()
        if a and a.lower() not in seen:
            seen.add(a.lower())
            out.append(a)
    return out

# Sanity check on 10 examples
print(f"{'Question':<55} {'Predicted':<25} {'Ground Truth (first)'}")
print("─" * 110)
for ex in trivia_ds.select(range(10)):
    question   = ex["question"]
    references = get_answers(ex)
    predicted  = answer_gpt2(question)
    print(f"{question[:54]:<55} {predicted[:24]:<25} {references[0][:24]}")

In [ ]:
# ── Shared scoring utilities (normalization, answer extraction, EM/F1) ───────
# Centralised so Task 2/3/4 and the Main Problem all score consistently.
# Fixes the original strict string-equality matcher that counted correct
# answers as misses (e.g. "A Boojum" vs "boojum", "Highway 61" vs "61").
import re, string

_ARTICLES = re.compile(r"\b(a|an|the)\b", re.IGNORECASE)
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)


def normalize_answer(s) -> str:
    """SQuAD-style normalisation: lowercase, strip punctuation/articles, collapse spaces."""
    if s is None:
        return ""
    s = str(s).lower()
    s = s.translate(_PUNCT_TABLE)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


_LEADINS = re.compile(
    r"^(the\s+)?(final\s+answer|correct\s+answer|answer)\s*(is|:)?\s*",
    re.IGNORECASE,
)


def clean_short_answer(text) -> str:
    """Reduce a possibly-verbose model output to a short answer phrase.

    Strips <think> reasoning, an explicit 'Answer:' line, keeps the first line,
    and removes 'The answer is ...' lead-ins. Use before scoring/TER so that
    short references are not compared against full sentences.
    """
    if not text:
        return ""
    raw = str(text).strip()
    if "</think>" in raw:                                    # drop chain-of-thought block
        raw = raw.split("</think>")[-1].strip()
    m = re.search(r"answer\s*:\s*(.+)", raw, re.IGNORECASE)  # prefer an 'Answer:' line
    if m:
        raw = m.group(1).strip()
    lines = raw.splitlines()
    if lines:
        raw = lines[0].strip()                               # keep only the first line
    raw = _LEADINS.sub("", raw).strip()                      # strip 'The answer is' lead-in
    return raw.strip().strip(".").strip()


def _subseq(a, b) -> bool:
    """True if list a is a contiguous sublist of list b."""
    if not a or len(a) > len(b):
        return False
    return any(b[i:i + len(a)] == a for i in range(len(b) - len(a) + 1))


def answer_matches(pred, refs, relaxed: bool = True) -> bool:
    """Normalized exact match against any reference; optional token-containment.

    relaxed=True also accepts the token-normalized reference being a contiguous span
    of the prediction or vice-versa (e.g. 'Highway 61' vs '61', 'Coventry, Warwickshire'
    vs 'Coventry'). Token-level, so '1961' never spuriously matches '61'.
    """
    np_ = normalize_answer(pred)
    if not np_:
        return False
    pt = np_.split()
    for r in refs:
        nr = normalize_answer(r)
        if not nr:
            continue
        if np_ == nr:
            return True
        if relaxed:
            rt = nr.split()
            if _subseq(rt, pt) or _subseq(pt, rt):
                return True
    return False


def exact_match(pred, refs) -> bool:
    """Backwards-compatible name used throughout the notebook (relaxed match)."""
    return answer_matches(pred, refs, relaxed=True)


def squad_f1(pred, refs) -> float:
    """Token-level F1 against the best reference (SQuAD-style)."""
    pt = normalize_answer(pred).split()
    best = 0.0
    for r in refs:
        rt = normalize_answer(r).split()
        if not pt and not rt:
            best = max(best, 1.0)
            continue
        if not pt or not rt:
            continue
        common = {t: min(pt.count(t), rt.count(t)) for t in set(pt) if t in rt}
        num_same = sum(common.values())
        if num_same == 0:
            continue
        prec, rec = num_same / len(pt), num_same / len(rt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


print("Scoring utilities ready: normalize_answer, clean_short_answer, exact_match (relaxed), squad_f1")

### Task 2 — Comparing Multiple Language Models on TriviaQA

Now that GPT-2 provides a baseline, we compare it against a wider set of models on the same 20 TriviaQA questions:

- **Local HuggingFace models** (SmolLM2-135M-Instruct, TinyLlama-1.1B-Chat, Qwen3.5-0.8B, DeepSeek-R1-Distill-1.5B) — instruction-tuned models loaded and run directly on the machine. A shared system prompt instructs each model to reply with a short phrase only.
- **IAedu API models** (GPT-5.5 and Claude Opus 4.7) — large frontier models accessed via the IAedu platform's streaming API, using credentials stored in a local `.env` file (never committed to git).

The final cell prints a side-by-side table with each model's prediction and an exact-match accuracy score at the bottom.

In [ ]:
import sys, os, json, uuid, requests
!{sys.executable} -m pip install python-dotenv -q

from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path(".") / ".env")

IAEDU_GPT_ENDPOINT      = os.getenv("IAEDU_GPT_ENDPOINT", "")
IAEDU_GPT_KEY           = os.getenv("IAEDU_GPT_KEY", "")
IAEDU_GPT_CHANNEL_ID    = os.getenv("IAEDU_GPT_CHANNEL_ID", "")
IAEDU_CLAUDE_ENDPOINT   = os.getenv("IAEDU_CLAUDE_ENDPOINT", "")
IAEDU_CLAUDE_KEY        = os.getenv("IAEDU_CLAUDE_KEY", "")
IAEDU_CLAUDE_CHANNEL_ID = os.getenv("IAEDU_CLAUDE_CHANNEL_ID", "")

SYSTEM_PROMPT = (
    "You are a factual question answering assistant. "
    "Answer with a short phrase or entity name only - no explanations, no full sentences."
)

def answer_iaedu(question: str, endpoint: str, api_key: str, channel_id: str) -> str:
    """Call an IAedu streaming agent endpoint (multipart/form-data, NDJSON response)."""
    if not endpoint or not api_key or channel_id.startswith("FILL"):
        return "[SKIPPED - not configured]"
    headers = {"x-api-key": api_key}
    files = {
        "channel_id": (None, channel_id),
        "thread_id":  (None, str(uuid.uuid4())),
        "user_info":  (None, "{}"),
        "message":    (None, f"{SYSTEM_PROMPT}\n\n{question}"),
    }
    try:
        resp = requests.post(endpoint, headers=headers, files=files, stream=True, timeout=30)
        resp.raise_for_status()
        tokens = []
        for raw in resp.iter_lines():
            if not raw:
                continue
            obj = json.loads(raw.decode("utf-8"))
            if obj.get("type") == "token":
                tokens.append(obj.get("content", ""))
            elif obj.get("type") == "done":
                break
        answer = "".join(tokens).strip()
        return answer.split("\n")[0].strip()
    except Exception as e:
        return f"[ERROR: {e}]"

# Connectivity test
print("GPT-5.5:        ", answer_iaedu("What is the capital of France?",
    IAEDU_GPT_ENDPOINT, IAEDU_GPT_KEY, IAEDU_GPT_CHANNEL_ID))
print("Claude Opus 4.7:", answer_iaedu("What is the capital of France?",
    IAEDU_CLAUDE_ENDPOINT, IAEDU_CLAUDE_KEY, IAEDU_CLAUDE_CHANNEL_ID))

In [ ]:
# ── Task 2 · Load local models (small baselines + one strong 7-12B model) ────
# Small instruction-tuned baselines are loaded in float16; one strong instruct model
# is loaded in 4-bit (bitsandbytes nf4) so the whole notebook fits on a Colab T4
# (~15 GB). The strong model is used for Task 3/4 and the spoken-QA pipeline.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_HAS_CUDA = torch.cuda.is_available()
_dtype = torch.float16 if _HAS_CUDA else torch.float32


def _model_device(mdl):
    """Device of a model's first parameter (handles 4-bit / device_map='auto')."""
    try:
        return next(mdl.parameters()).device
    except Exception:
        return torch.device("cuda" if _HAS_CUDA else "cpu")


SMALL_MODELS = {
    "SmolLM2-135M":     "HuggingFaceTB/SmolLM2-135M-Instruct",
    "TinyLlama-1.1B":   "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen3.5-0.8B":     "Qwen/Qwen3.5-0.8B",
    "DeepSeek-R1-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
}

# Strong model for Task 3/4 + pipeline. Default fits a Colab T4 alongside the
# baselines. On a 24 GB+ GPU (e.g. RTX 5090 / A100) switch BIG_MODEL_ID to a 12B —
# e.g. "google/gemma-3-12b-it" (gated: needs `huggingface-cli login` / HF_TOKEN) —
# and/or set LOAD_SMALL=False to free room.
BIG_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
BIG_MODEL_KEY = "Qwen2.5-7B-Instruct"
LOAD_SMALL    = True     # set False to load only the strong model (saves memory)

loaded_models = {}       # label -> (tokenizer, model)

if LOAD_SMALL:
    for label, hf_id in SMALL_MODELS.items():
        print(f"Loading {label}...", end=" ", flush=True)
        tok = AutoTokenizer.from_pretrained(hf_id)
        mdl = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=_dtype)
        mdl = mdl.to("cuda") if _HAS_CUDA else mdl
        mdl.eval()
        loaded_models[label] = (tok, mdl)
        print("done")

# Strong model: 4-bit on GPU, fp32 fallback on CPU.
print(f"Loading {BIG_MODEL_KEY} ({BIG_MODEL_ID})...", end=" ", flush=True)
try:
    btok = AutoTokenizer.from_pretrained(BIG_MODEL_ID)
    if _HAS_CUDA:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
        bmdl = AutoModelForCausalLM.from_pretrained(
            BIG_MODEL_ID, quantization_config=bnb, device_map="auto")
    else:
        bmdl = AutoModelForCausalLM.from_pretrained(BIG_MODEL_ID, torch_dtype=torch.float32)
    bmdl.eval()
    loaded_models[BIG_MODEL_KEY] = (btok, bmdl)
    PIPELINE_MODEL_KEY = BIG_MODEL_KEY
    print("done")
except Exception as e:
    print(f"FAILED ({str(e)[:70]})")
    PIPELINE_MODEL_KEY = ("Qwen3.5-0.8B" if "Qwen3.5-0.8B" in loaded_models
                          else (next(iter(loaded_models)) if loaded_models else None))

print(f"\nLoaded models : {list(loaded_models)}")
print(f"Pipeline model: {PIPELINE_MODEL_KEY}")


def answer_hf(question: str, tok, mdl, max_new_tokens: int = 64) -> str:
    """Answer a question using any instruction-tuned HF model (device-aware)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    if tok.chat_template is not None:
        try:
            input_text = tok.apply_chat_template(
                messages, enable_thinking=False, add_generation_prompt=True, tokenize=False)
        except TypeError:
            input_text = tok.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False)
    else:
        input_text = f"System: {SYSTEM_PROMPT}\nUser: {question}\nAssistant:"

    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tok.eos_token_id)
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()

    if "</think>" in raw:                       # DeepSeek-R1 reasoning block
        raw = raw.split("</think>")[-1].strip()
    for sep in ("\n", ". "):                    # keep only the first line / sentence
        idx = raw.find(sep)
        if idx != -1 and idx < 120:
            raw = raw[:idx]
    return raw.strip()


def free_small_models():
    """Free the small baseline LLMs from VRAM, keeping the pipeline model, so a
    12B model + (4-bit) VibeVoice fit a 15 GB T4 for the complete pipeline."""
    import gc
    for _k in [k for k in loaded_models if k != PIPELINE_MODEL_KEY]:
        del loaded_models[_k]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Freed baselines; kept pipeline model {PIPELINE_MODEL_KEY}. "
          f"Loaded now: {list(loaded_models)}")

In [ ]:
# ── Task 2 · Side-by-side model comparison on 20 TriviaQA examples ───────────
# Scoring uses the shared utilities (normalize_answer / exact_match / squad_f1).
N_COMPARE = 20
sample = trivia_ds.select(range(N_COMPARE))

local_names = ["GPT-2"] + list(loaded_models.keys())

api_configs = [
    ("GPT-5.5",         IAEDU_GPT_ENDPOINT,    IAEDU_GPT_KEY,    IAEDU_GPT_CHANNEL_ID),
    ("Claude-Opus-4.7", IAEDU_CLAUDE_ENDPOINT, IAEDU_CLAUDE_KEY, IAEDU_CLAUDE_CHANNEL_ID),
]
api_names   = [name for name, ep, _, ch in api_configs if ep and not ch.startswith("FILL")]
model_names = local_names + api_names

all_preds = {name: [] for name in model_names}
for ex in sample:
    q = ex["question"]
    all_preds["GPT-2"].append(answer_gpt2(q))
    for label, (tok, mdl) in loaded_models.items():
        all_preds[label].append(answer_hf(q, tok, mdl))
    for name, ep, key, ch in api_configs:
        if ep and not ch.startswith("FILL"):
            all_preds[name].append(answer_iaedu(q, ep, key, ch))

COL, Q = 20, 36
header = f"{'Question':<{Q}}" + "".join(f"{n[:COL-1]:<{COL}}" for n in model_names) + "  Reference"
print(header)
print("─" * len(header))

scores = {name: 0 for name in model_names}
f1s    = {name: 0.0 for name in model_names}
for i, ex in enumerate(sample):
    refs = get_answers(ex)
    row  = f"{ex['question'][:Q-1]:<{Q}}"
    for name in model_names:
        pred = clean_short_answer(all_preds[name][i])
        if exact_match(pred, refs):
            scores[name] += 1
        f1s[name] += squad_f1(pred, refs)
        row += f"{pred[:COL-1]:<{COL}}"
    row += f"  {refs[0][:24]}"
    print(row)

print("─" * len(header))
acc_row = f"{'Exact-match hits':<{Q}}"
f1_row  = f"{'Mean token-F1':<{Q}}"
for name in model_names:
    acc_row += f"{scores[name]}/{N_COMPARE}{'':>{COL - len(str(scores[name])) - len(str(N_COMPARE)) - 1}}"
    f1_row  += f"{f1s[name] / N_COMPARE:<{COL}.2f}"
print(acc_row)
print(f1_row)

# Save the 20-example predictions (all models, untruncated) for inspection / judging.
import json
json.dump([dict(question=sample[i]["question"], references=get_answers(sample[i]),
                **{n: all_preds[n][i] for n in model_names}) for i in range(N_COMPARE)],
          open("predictions_trivia20.json", "w"), ensure_ascii=False, indent=1)
print("Saved predictions_trivia20.json")

In [ ]:
# ── DeepSeek-R1 reasoning model — a FAIR re-test ─────────────────────────────
# In Task 2 it scored 0/20, but ONLY because answer_hf caps generation at ~64 tokens —
# a reasoning model needs room to finish its <think>...</think> block first. Here we give
# it 512 tokens, strip the reasoning trace, and extract the final short answer.
import torch
_DS_KEY = "DeepSeek-R1-1.5B"
if _DS_KEY in loaded_models:
    _ds_tok, _ds_mdl = loaded_models[_DS_KEY]
else:
    from transformers import AutoTokenizer as _AT, AutoModelForCausalLM as _AM
    _ds_tok = _AT.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    _ds_mdl = _AM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
                                  torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    _ds_mdl = _ds_mdl.to("cuda") if torch.cuda.is_available() else _ds_mdl
    _ds_mdl.eval()

def answer_deepseek(question, max_new_tokens=512):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    try:
        text = _ds_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    except Exception:
        text = f"{SYSTEM_PROMPT}\n\n{question}"
    inp = _ds_tok(text, return_tensors="pt").to(_model_device(_ds_mdl))
    with torch.no_grad():
        out = _ds_mdl.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=_ds_tok.eos_token_id)
    raw = _ds_tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
    if "</think>" in raw:                       # drop the reasoning trace, keep the answer
        raw = raw.split("</think>")[-1]
    return clean_short_answer(raw)

_ds_sample = trivia_ds.select(range(20))
_ds_em = _ds_f1 = 0.0
for ex in _ds_sample:
    refs = get_answers(ex)
    pred = answer_deepseek(ex["question"])
    _ds_em += exact_match(pred, refs)
    _ds_f1 += squad_f1(pred, refs)
_n = len(_ds_sample)
print("DeepSeek-R1-1.5B — FAIR re-test (512 tokens, <think> stripped):")
print(f"  EM {int(_ds_em)}/{_n}   F1 {_ds_f1 / _n:.2f}   (was 0/20 when truncated at 64 tokens)")

import json as _json
_json.dump({"em": int(_ds_em), "f1": round(_ds_f1 / _n, 4), "n": _n, "max_new_tokens": 512},
           open("results_deepseek_fair.json", "w"), indent=1)
print("  saved results_deepseek_fair.json")

In [ ]:
# ── (Experiment) How does a Gemma model compare on the same 20 questions? ────
# Self-contained + FAIL-SOFT: loads a Gemma in 4-bit, scores the same 20 TriviaQA
# items as Task 2, then frees it so later cells are unaffected. If it can't load
# (gating / disk / OOM / unknown ID) it skips cleanly and the pipeline keeps Qwen.
#
# Notes: Gemma 4 (Apache-2.0) is ungated; Gemma 3 is gated (accept its license on the
# HF page + run `huggingface-cli login`). The 12B variants are MULTIMODAL (~24 GB
# download) and will likely OOM on a 15 GB T4 next to the other models — best on a
# 24 GB GPU, or use a smaller id below.
import gc
import torch
import transformers as _tf
from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

try:
    free_small_models()   # make room for Gemma on a 15 GB T4
except Exception:
    pass

GEMMA_CANDIDATES = ["google/gemma-4-12B-it", "google/gemma-3-12b-it", "google/gemma-3-4b-it"]


def _gemma_load(mid):
    """Return (kind, handler, model) trying text then multimodal classes; None on failure."""
    if torch.cuda.is_available():
        kw = dict(quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
            device_map="auto")
    else:
        kw = dict(torch_dtype=torch.float32)
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        mdl = AutoModelForCausalLM.from_pretrained(mid, **kw).eval()
        return ("causal", tok, mdl)
    except Exception:
        for cls in ("AutoModelForImageTextToText", "Gemma3ForConditionalGeneration",
                    "AutoModelForMultimodalLM"):
            if not hasattr(_tf, cls):
                continue
            try:
                proc = AutoProcessor.from_pretrained(mid)
                mdl = getattr(_tf, cls).from_pretrained(mid, **kw).eval()
                return ("multimodal", proc, mdl)
            except Exception:
                continue
    return None


def _gemma_answer(kind, h, mdl, question):
    msg = f"{SYSTEM_PROMPT}\n\nQuestion: {question}"
    if kind == "multimodal":
        conv = [{"role": "user", "content": [{"type": "text", "text": msg}]}]
        inp = h.apply_chat_template(conv, add_generation_prompt=True, tokenize=True,
                                    return_dict=True, return_tensors="pt").to(mdl.device)
        ilen = inp["input_ids"].shape[-1]
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=64, do_sample=False)
        return h.batch_decode(out[:, ilen:], skip_special_tokens=True)[0]
    text = h.apply_chat_template([{"role": "user", "content": msg}],
                                 add_generation_prompt=True, tokenize=False)
    inp = h(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=64, do_sample=False, pad_token_id=h.eos_token_id)
    return h.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)


_loaded, GEMMA_USED = None, None
for _mid in GEMMA_CANDIDATES:
    print(f"Loading {_mid} (4-bit)...", end=" ", flush=True)
    try:
        _loaded = _gemma_load(_mid)
    except Exception as e:
        print(f"failed ({str(e)[:50]})"); continue
    if _loaded:
        GEMMA_USED = _mid; print("OK"); break
    print("could not load")

if _loaded:
    _kind, _h, _gmdl = _loaded
    _gs = trivia_ds.select(range(20))
    g_em = g_f1 = 0.0
    print(f"\n{'Question':<46}{'Gemma':<24}{'Reference'}")
    print("─" * 92)
    for ex in _gs:
        refs = get_answers(ex)
        try:
            pred = clean_short_answer(_gemma_answer(_kind, _h, _gmdl, ex["question"]))
        except Exception as e:
            pred = f"[ERR {str(e)[:15]}]"
        g_em += exact_match(pred, refs)
        g_f1 += squad_f1(pred, refs)
        print(f"{ex['question'][:45]:<46}{pred[:23]:<24}{refs[0][:24]}")
    n = len(_gs)
    print("─" * 92)
    print(f"Gemma  ({GEMMA_USED}):  EM {int(g_em)}/{n}   F1 {g_f1 / n:.2f}")
    try:
        print(f"Qwen2.5-7B-Instruct (Task 2):  EM {scores.get('Qwen2.5-7B-Instruct', '?')}/{n}"
              f"   F1 {f1s.get('Qwen2.5-7B-Instruct', 0) / n:.2f}")
    except Exception:
        pass
    print("\nIf Gemma clearly wins, we can promote it to the pipeline model (a larger change).")
    del _gmdl, _loaded                      # free VRAM so later cells are unaffected
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("\nGemma not available here (gating / disk / OOM / id). Keeping Qwen2.5-7B-Instruct.")
    print("Tips: Gemma-3 is gated → `huggingface-cli login` + accept its license; for low")
    print("disk/VRAM use google/gemma-3-4b-it; the 12B is multimodal and ~24 GB to download.")

### Task 3 — Strategies for Improving Question Answering

We test four strategies — individually and combined — to see which improves exact-match accuracy over the Task 2 baseline. All strategies are evaluated on the same 20 TriviaQA examples using Qwen3.5-0.8B (best local model) and both IAedu API models.

| Strategy | What changes |
|---|---|
| **S1 — Zero-shot (baseline)** | Plain system prompt asking for a short answer |
| **S2 — Three-shot** | Three solved examples from different domains added to anchor the format |
| **S3 — Chain-of-thought** | Model is asked to reason step by step; final answer extracted after `Answer:` |
| **S4 — RAG (Wikipedia)** | A Wikipedia summary is fetched per question and prepended as context |
| **S2+S4 — One-shot + RAG** | Combines the example anchor with retrieved context |
| **S3+S4 — CoT + RAG** | Combines reasoning with retrieved context |

In [ ]:
import sys
!{sys.executable} -m pip install wikipedia -q

# ── Task 3 · Strategy helpers ─────────────────────────────────────────────────
import re, time, wikipedia

# Three diverse examples across different TriviaQA domains (science, history, art)
FEW_SHOT_EXAMPLES = (
    "Q: What is the chemical symbol for gold?\n"
    "A: Au\n\n"
    "Q: In which year did the Berlin Wall fall?\n"
    "A: 1989\n\n"
    "Q: Who painted the Sistine Chapel ceiling?\n"
    "A: Michelangelo\n\n"
)

COT_SYSTEM = (
    "You are a factual QA assistant. Think step by step, then write your final answer "
    "on the last line starting with exactly 'Answer:' followed by a short phrase or name only."
)

SHORT_SYSTEM = (
    "You are a factual QA assistant. "
    "Answer with a short phrase or entity name only - no explanations."
)

def fetch_wiki_context(question: str, max_chars: int = 1500, top_k: int = 3) -> str:
    # Search Wikipedia and concatenate summaries of the top-k results
    try:
        titles = wikipedia.search(question, results=top_k)
    except Exception:
        return ""
    summaries = []
    for title in titles:
        try:
            summaries.append(wikipedia.summary(title, sentences=5, auto_suggest=False))
        except wikipedia.DisambiguationError as e:
            try:
                summaries.append(wikipedia.summary(e.options[0], sentences=5, auto_suggest=False))
            except Exception:
                pass
        except Exception:
            pass
    return "\n\n".join(summaries)[:max_chars]

def extract_cot_answer(raw: str) -> str:
    # Extract text after 'Answer:' in a chain-of-thought response
    match = re.search(r"Answer:\s*(.+)", raw, re.IGNORECASE)
    if match:
        return match.group(1).strip().split("\n")[0].strip()
    lines = [l.strip() for l in raw.strip().splitlines() if l.strip()]
    return lines[-1] if lines else raw.strip()

# ── Strategy wrappers for local HF models ────────────────────────────────────
def s1_zero_shot(question, tok, mdl):
    return answer_hf(question, tok, mdl)

def s2_one_shot(question, tok, mdl):
    user_msg = FEW_SHOT_EXAMPLES + f"Q: {question}\nA:"
    messages = [{"role": "system", "content": SHORT_SYSTEM},
                {"role": "user",   "content": user_msg}]
    try:
        input_text = tok.apply_chat_template(messages, enable_thinking=False,
                                             add_generation_prompt=True, tokenize=False)
    except TypeError:
        input_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with __import__("torch").no_grad():
        out = mdl.generate(**inputs, max_new_tokens=30, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    return raw.split("\n")[0].strip()

def s3_cot(question, tok, mdl):
    messages = [{"role": "system", "content": COT_SYSTEM},
                {"role": "user",   "content": question}]
    try:
        input_text = tok.apply_chat_template(messages, enable_thinking=False,
                                             add_generation_prompt=True, tokenize=False)
    except TypeError:
        input_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with __import__("torch").no_grad():
        out = mdl.generate(**inputs, max_new_tokens=120, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    return extract_cot_answer(raw)

def s4_rag(question, tok, mdl):
    ctx = fetch_wiki_context(question)
    user_msg = (f"Context: {ctx}\n\nQuestion: {question}" if ctx else question)
    messages = [{"role": "system", "content": SHORT_SYSTEM},
                {"role": "user",   "content": user_msg}]
    try:
        input_text = tok.apply_chat_template(messages, enable_thinking=False,
                                             add_generation_prompt=True, tokenize=False)
    except TypeError:
        input_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with __import__("torch").no_grad():
        out = mdl.generate(**inputs, max_new_tokens=30, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    return raw.split("\n")[0].strip()

def s2s4_one_shot_rag(question, tok, mdl):
    ctx = fetch_wiki_context(question)
    user_msg = FEW_SHOT_EXAMPLES + (f"Context: {ctx}\n\nQ: {question}\nA:" if ctx
                                    else f"Q: {question}\nA:")
    messages = [{"role": "system", "content": SHORT_SYSTEM},
                {"role": "user",   "content": user_msg}]
    try:
        input_text = tok.apply_chat_template(messages, enable_thinking=False,
                                             add_generation_prompt=True, tokenize=False)
    except TypeError:
        input_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with __import__("torch").no_grad():
        out = mdl.generate(**inputs, max_new_tokens=30, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    return raw.split("\n")[0].strip()

def s3s4_cot_rag(question, tok, mdl):
    ctx = fetch_wiki_context(question)
    user_msg = (f"Context: {ctx}\n\nQuestion: {question}" if ctx else question)
    messages = [{"role": "system", "content": COT_SYSTEM},
                {"role": "user",   "content": user_msg}]
    try:
        input_text = tok.apply_chat_template(messages, enable_thinking=False,
                                             add_generation_prompt=True, tokenize=False)
    except TypeError:
        input_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_text, return_tensors="pt").to(_model_device(mdl))
    with __import__("torch").no_grad():
        out = mdl.generate(**inputs, max_new_tokens=120, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    return extract_cot_answer(raw)

# ── Robust IAedu caller (retry + backoff) ─────────────────────────────────────
def _post_iaedu(message, endpoint, api_key, channel_id, retries=4, base_delay=1.5):
    # Low-level IAedu call with exponential backoff
    if not endpoint or not api_key or channel_id.startswith("FILL"):
        return "[SKIPPED]"
    import uuid as _uuid
    headers = {"x-api-key": api_key}
    last_err = "no response"
    for attempt in range(retries):
        files = {
            "channel_id": (None, channel_id),
            "thread_id":  (None, str(_uuid.uuid4())),
            "user_info":  (None, "{}"),
            "message":    (None, message),
        }
        try:
            resp = requests.post(endpoint, headers=headers, files=files, stream=True, timeout=60)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"status {resp.status_code}")
            resp.raise_for_status()
            tokens = []
            for raw in resp.iter_lines():
                if not raw:
                    continue
                try:
                    obj = json.loads(raw.decode("utf-8"))
                except Exception:
                    continue
                if obj.get("type") == "token":
                    tokens.append(obj.get("content", ""))
                elif obj.get("type") == "done":
                    break
            text = "".join(tokens).strip()
            if text:
                return text
            last_err = "empty response"
        except Exception as e:
            last_err = str(e)
        time.sleep(base_delay * (2 ** attempt))
    return f"[ERROR: {last_err}]"

def _raw_iaedu(full_message, endpoint, api_key, channel_id):
    return _post_iaedu(full_message, endpoint, api_key, channel_id)

# ── Strategy wrappers for IAedu API models ────────────────────────────────────
def api_s1(question, ep, key, ch):
    return clean_short_answer(_post_iaedu(f"{SHORT_SYSTEM}\n\n{question}", ep, key, ch))

def api_s2(question, ep, key, ch):
    msg = f"{SHORT_SYSTEM}\n\n{FEW_SHOT_EXAMPLES}Q: {question}\nA:"
    return clean_short_answer(_post_iaedu(msg, ep, key, ch))

def api_s3(question, ep, key, ch):
    msg = f"{COT_SYSTEM}\n\nQuestion: {question}"
    return clean_short_answer(extract_cot_answer(_post_iaedu(msg, ep, key, ch)))

def api_s4(question, ep, key, ch):
    ctx = fetch_wiki_context(question)
    msg = (f"{SHORT_SYSTEM}\n\nContext: {ctx}\n\nQuestion: {question}" if ctx
           else f"{SHORT_SYSTEM}\n\n{question}")
    return clean_short_answer(_post_iaedu(msg, ep, key, ch))

def api_s2s4(question, ep, key, ch):
    ctx = fetch_wiki_context(question)
    msg = (f"{SHORT_SYSTEM}\n\n{FEW_SHOT_EXAMPLES}Context: {ctx}\n\nQ: {question}\nA:" if ctx
           else f"{SHORT_SYSTEM}\n\n{FEW_SHOT_EXAMPLES}Q: {question}\nA:")
    return clean_short_answer(_post_iaedu(msg, ep, key, ch))

def api_s3s4(question, ep, key, ch):
    ctx = fetch_wiki_context(question)
    msg = (f"{COT_SYSTEM}\n\nContext: {ctx}\n\nQuestion: {question}" if ctx
           else f"{COT_SYSTEM}\n\nQuestion: {question}")
    return clean_short_answer(extract_cot_answer(_post_iaedu(msg, ep, key, ch)))

print("Strategy helpers ready (3-shot examples, Wikipedia search RAG with top-3 pages).")


In [ ]:
# ── Task 3 · Evaluate all strategies × all models on 20 TriviaQA examples ────
import time
N_EVAL = 20
eval_sample = trivia_ds.select(range(N_EVAL))

STRATEGIES = ["S1-ZeroShot", "S2-OneShot", "S3-CoT", "S4-RAG", "S2+S4", "S3+S4"]

# Best local model from Task 2 (fallback to the first loaded model if absent)
_best_local = globals().get("PIPELINE_MODEL_KEY") or ("Qwen3.5-0.8B" if "Qwen3.5-0.8B" in loaded_models else next(iter(loaded_models)))
EVAL_LOCAL = {_best_local: loaded_models[_best_local]}

EVAL_API = [
    ("GPT-5.5",         IAEDU_GPT_ENDPOINT,    IAEDU_GPT_KEY,    IAEDU_GPT_CHANNEL_ID),
    ("Claude-Opus-4.7", IAEDU_CLAUDE_ENDPOINT, IAEDU_CLAUDE_KEY, IAEDU_CLAUDE_CHANNEL_ID),
]

local_strategy_fns = [s1_zero_shot, s2_one_shot, s3_cot, s4_rag, s2s4_one_shot_rag, s3s4_cot_rag]
api_strategy_fns   = [api_s1, api_s2, api_s3, api_s4, api_s2s4, api_s3s4]

# Collect predictions: results[model][strategy] = [pred, ...]
results = {}

for label, (tok, mdl) in EVAL_LOCAL.items():
    results[label] = {s: [] for s in STRATEGIES}
    for ex in eval_sample:
        q = ex["question"]
        for strat, fn in zip(STRATEGIES, local_strategy_fns):
            results[label][strat].append(fn(q, tok, mdl))
    print(f"[done] {label}")

for name, ep, key, ch in EVAL_API:
    if not ep or ch.startswith("FILL"):
        continue
    results[name] = {s: [] for s in STRATEGIES}
    for ex in eval_sample:
        q = ex["question"]
        for strat, fn in zip(STRATEGIES, api_strategy_fns):
            results[name][strat].append(fn(q, ep, key, ch))
            time.sleep(0.3)        # throttle to stay under API rate limits
    print(f"[done] {name}")

# Surface API failures instead of silently scoring them as misses
for name in results:
    bad = sum(1 for s in STRATEGIES for p in results[name][s]
              if (not p) or str(p).startswith("[ERROR") or str(p).startswith("[SKIPPED"))
    if bad:
        print(f"  ! {name}: {bad} empty/error predictions (likely rate-limited — re-run or raise delay)")

# ── Compute exact-match scores (predictions cleaned before scoring) ───────────
scores = {}  # model -> strategy -> int
for model_name, strat_preds in results.items():
    scores[model_name] = {}
    for strat, preds in strat_preds.items():
        hits = sum(exact_match(clean_short_answer(preds[i]), get_answers(eval_sample[i]))
                   for i in range(N_EVAL))
        scores[model_name][strat] = hits

# ── Print results table ───────────────────────────────────────────────────────
model_names_t3 = list(results.keys())
COL_W = 16
header = f"{'Strategy':<14}" + "".join(f"{m[:COL_W-1]:<{COL_W}}" for m in model_names_t3)
print("\n" + header)
print("─" * len(header))
for strat in STRATEGIES:
    row = f"{strat:<14}"
    for m in model_names_t3:
        row += f"{str(scores[m][strat]) + '/' + str(N_EVAL):<{COL_W}}"
    print(row)
print("─" * len(header))

# Best strategy per model
for m in model_names_t3:
    best_s = max(STRATEGIES, key=lambda s: scores[m][s])
    print(f"  Best for {m}: {best_s}  ({scores[m][best_s]}/{N_EVAL})")

### Task 4 — TER Evaluation on 500 TriviaQA Examples

We pick the best (model, strategy) combination from Task 3 and scale up to the first 500 validation examples. The [TER metric](https://github.com/huggingface/evaluate/tree/main/metrics/ter) (Translation Edit Rate) measures the minimum number of edits — insertions, deletions, substitutions, and shifts — needed to convert the hypothesis into the reference, normalised by the reference length. A lower TER means a better match.

Since calling the IAedu API for 500 examples would be slow and costly, we use the best **local** model/strategy identified in Task 3 for this bulk evaluation.

In [ ]:
# ── Task 4 · TER (+ EM / F1) on first 500 TriviaQA validation examples ───────
import sys, time
!{sys.executable} -m pip install sacrebleu -q

from evaluate import load as load_metric

ter = load_metric("ter")

N_TER = 500
ter_sample = trivia_ds.select(range(N_TER))

# ── Choose which backends to evaluate ────────────────────────────────────────
# Set to True to also run the IAedu API models (slow — ~500 calls each, 1–2 min/model)
RUN_GPT    = True
RUN_CLAUDE = True

# ── Pick best local model + strategy from Task 3 ─────────────────────────────
local_scores = {(m, s): scores[m][s]
                for m in scores if m in loaded_models
                for s in STRATEGIES}

if local_scores:
    best_local_model, best_local_strat = max(local_scores, key=lambda k: local_scores[k])
else:
    best_local_model = next(iter(loaded_models))
    best_local_strat = "S1-ZeroShot"

print(f"Local model : {best_local_model} | {best_local_strat}  (best from Task 3)")

local_fn_map = dict(zip(STRATEGIES, local_strategy_fns))
api_fn_map   = dict(zip(STRATEGIES, api_strategy_fns))

# ── Helper: run one backend over all 500 examples ────────────────────────────
def run_backend(label, predict_fn, sample, n, is_api=False):
    predictions, references, records = [], [], []
    em_hits, f1_total = 0, 0.0
    errors = 0

    print(f"\n[{label}] Starting {n} predictions...")
    for i, ex in enumerate(sample):
        pred_raw = predict_fn(ex["question"])
        pred = clean_short_answer(pred_raw)

        # Detect and surface silent API failures immediately
        if is_api and (not pred_raw or str(pred_raw).startswith("[ERROR") or
                       str(pred_raw).startswith("[SKIPPED")):
            errors += 1
            print(f"  ! [{label}] Q{i+1} FAILED: {str(pred_raw)[:80]}")

        refs = get_answers(ex)
        _em  = exact_match(pred, refs)
        _f1  = squad_f1(pred, refs)
        em_hits   += int(_em)
        f1_total  += _f1
        predictions.append(pred)
        references.append(refs)
        records.append({"question": ex["question"], "prediction": pred,
                        "references": refs, "em": int(_em), "f1": round(_f1, 4)})

        if (i + 1) % 50 == 0:
            running_em = 100 * em_hits / (i + 1)
            err_str = f"  {errors} API errors so far" if is_api and errors else ""
            print(f"  {i+1}/{n} done  |  running EM {running_em:.1f}%{err_str}")

        if is_api:
            time.sleep(0.4)   # stay under rate limit

    if is_api and errors:
        print(f"  ! [{label}] Total failures: {errors}/{n}  "
              f"(likely rate-limited — re-run or increase sleep above)")

    return predictions, references, records, em_hits, f1_total

# ── TER helper ────────────────────────────────────────────────────────────────
def compute_ter(predictions, references):
    ter_hyp, ter_ref = [], []
    for p, refs in zip(predictions, references):
        rn = normalize_answer(refs[0]) if refs else ""
        if not rn:
            continue
        ter_hyp.append(normalize_answer(p) or "<blank>")
        ter_ref.append([rn])
    return ter.compute(predictions=ter_hyp, references=ter_ref)["score"]

# ── Print summary ─────────────────────────────────────────────────────────────
def print_summary(label, strategy, em_hits, f1_total, ter_score, n):
    print(f"\nResults — {label} | {strategy} — over {n} TriviaQA examples")
    print(f"  Exact-match : {em_hits}/{n}  ({100 * em_hits / n:.1f}%)")
    print(f"  Token-F1    : {f1_total / n:.3f}")
    print(f"  TER         : {ter_score:.2f}  (lower is better)")

# ═════════════════════════════════════════════════════════════════════════════
# 1) Local model
# ═════════════════════════════════════════════════════════════════════════════
tok_t4, mdl_t4 = loaded_models[best_local_model]
strat_fn_local = local_fn_map[best_local_strat]

preds_local, refs_local, records_local, em_local, f1_local = run_backend(
    best_local_model,
    lambda q: strat_fn_local(q, tok_t4, mdl_t4),
    ter_sample, N_TER, is_api=False
)

ter_local = compute_ter(preds_local, refs_local)
print_summary(best_local_model, best_local_strat, em_local, f1_local, ter_local, N_TER)

json.dump(records_local, open("predictions_trivia500.json", "w"), ensure_ascii=False, indent=1)
print(f"Saved {len(records_local)} predictions -> predictions_trivia500.json")

# ═════════════════════════════════════════════════════════════════════════════
# 2) GPT-5.5 via IAedu (optional)
# ═════════════════════════════════════════════════════════════════════════════
if RUN_GPT and IAEDU_GPT_ENDPOINT and not IAEDU_GPT_CHANNEL_ID.startswith("FILL"):
    # Use the best strategy found in Task 3 for GPT, defaulting to S4-RAG
    best_api_strat = max(
        (s for s in STRATEGIES if "GPT-5.5" in scores and scores["GPT-5.5"]),
        key=lambda s: scores.get("GPT-5.5", {}).get(s, 0),
        default="S4-RAG"
    )
    gpt_fn = api_fn_map[best_api_strat]

    preds_gpt, refs_gpt, records_gpt, em_gpt, f1_gpt = run_backend(
        "GPT-5.5",
        lambda q: gpt_fn(q, IAEDU_GPT_ENDPOINT, IAEDU_GPT_KEY, IAEDU_GPT_CHANNEL_ID),
        ter_sample, N_TER, is_api=True
    )
    ter_gpt = compute_ter(preds_gpt, refs_gpt)
    print_summary("GPT-5.5", best_api_strat, em_gpt, f1_gpt, ter_gpt, N_TER)
    json.dump(records_gpt, open("predictions_trivia500_gpt.json", "w"), ensure_ascii=False, indent=1)
    print(f"Saved predictions_trivia500_gpt.json")
elif RUN_GPT:
    print("\n[GPT-5.5] Skipped — endpoint or channel_id not configured.")

# ═════════════════════════════════════════════════════════════════════════════
# 3) Claude-Opus-4.7 via IAedu (optional)
# ═════════════════════════════════════════════════════════════════════════════
if RUN_CLAUDE and IAEDU_CLAUDE_ENDPOINT and not IAEDU_CLAUDE_CHANNEL_ID.startswith("FILL"):
    best_api_strat_cl = max(
        (s for s in STRATEGIES if "Claude-Opus-4.7" in scores and scores["Claude-Opus-4.7"]),
        key=lambda s: scores.get("Claude-Opus-4.7", {}).get(s, 0),
        default="S4-RAG"
    )
    cl_fn = api_fn_map[best_api_strat_cl]

    preds_cl, refs_cl, records_cl, em_cl, f1_cl = run_backend(
        "Claude-Opus-4.7",
        lambda q: cl_fn(q, IAEDU_CLAUDE_ENDPOINT, IAEDU_CLAUDE_KEY, IAEDU_CLAUDE_CHANNEL_ID),
        ter_sample, N_TER, is_api=True
    )
    ter_cl = compute_ter(preds_cl, refs_cl)
    print_summary("Claude-Opus-4.7", best_api_strat_cl, em_cl, f1_cl, ter_cl, N_TER)
    json.dump(records_cl, open("predictions_trivia500_claude.json", "w"), ensure_ascii=False, indent=1)
    print(f"Saved predictions_trivia500_claude.json")
elif RUN_CLAUDE:
    print("\n[Claude-Opus-4.7] Skipped — endpoint or channel_id not configured.")


In [ ]:
# ── LLM-as-judge (Qwen-7B, judged WITH the reference) over the 500 TriviaQA answers ──
# Reads predictions_trivia500.json (saved by the previous cell) so it can run without
# re-generating. The judge sees QUESTION + ACCEPTED ANSWER(S) + MODEL ANSWER and decides
# semantic equivalence — it is NOT asked to fact-check from its own knowledge, which makes
# it far more reliable on niche trivia. Caveat: it's the same model family that produced
# the answers, so it can miss aliases it doesn't know (e.g. "David Seville" = "Ross
# Bagdasarian"); hand-check a handful. Cross-tabbing judge vs EM splits the EM "failures"
# into metric artifacts (right but worded differently) vs genuine errors.
import json, re, torch

JUDGE_KEY = (globals().get("PIPELINE_MODEL_KEY")
             or ("Qwen2.5-7B-Instruct" if "Qwen2.5-7B-Instruct" in loaded_models
                 else next(iter(loaded_models))))
_jtok, _jmdl = loaded_models[JUDGE_KEY]
print(f"Judge model: {JUDGE_KEY}")

_JUDGE_SYS = ("You grade trivia answers. The MODEL ANSWER is correct if it refers to the same "
              "entity or fact as ANY accepted answer, ignoring case, wording, word order, or "
              "extra surrounding words. Reply with exactly one word: YES or NO.")

def judge_correct(question, prediction, refs):
    refstr = " / ".join(str(r) for r in refs[:6]) if refs else "(none)"
    user = (f"Question: {question}\nAccepted answer(s): {refstr}\n"
            f"Model answer: {prediction}\nIs the model answer correct?")
    msgs = [{"role": "system", "content": _JUDGE_SYS}, {"role": "user", "content": user}]
    try:
        text = _jtok.apply_chat_template(msgs, enable_thinking=False,
                                         add_generation_prompt=True, tokenize=False)
    except TypeError:
        text = _jtok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = _jtok(text, return_tensors="pt").to(_model_device(_jmdl))
    with torch.no_grad():
        out = _jmdl.generate(**inp, max_new_tokens=4, do_sample=False,
                             pad_token_id=_jtok.eos_token_id)
    resp = _jtok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip().lower()
    return resp.startswith("y") or bool(re.match(r"\W*yes", resp))

data = json.load(open("predictions_trivia500.json"))
JUDGE_N = len(data)                 # lower this (e.g. 100) to go faster
A = B = C = D = 0                   # EMhit/Y, EMmiss/Y, EMmiss/N, EMhit/N
for i, d in enumerate(data[:JUDGE_N]):
    em = bool(d["em"])
    jc = judge_correct(d["question"], d["prediction"], d["references"])
    if   em and jc:           A += 1
    elif (not em) and jc:     B += 1
    elif (not em) and not jc: C += 1
    else:                     D += 1
    if (i + 1) % 50 == 0:
        print(f"  judged {i + 1}/{JUDGE_N}")

n = A + B + C + D
print(f"\nLLM-as-judge over {n} TriviaQA answers (judge = {JUDGE_KEY}, with reference)")
print(f"  EM accuracy:    {A + D}/{n}  ({100 * (A + D) / n:.1f}%)")
print(f"  Judge accuracy: {A + B}/{n}  ({100 * (A + B) / n:.1f}%)")
print( "  ---- EM vs judge cross-tab ----")
print(f"  EM-hit  & judge-correct (clean correct):   {A}")
print(f"  EM-miss & judge-correct (wording/alias):   {B}   <- metric was unfair here")
print(f"  EM-miss & judge-wrong   (genuine error):   {C}")
print(f"  EM-hit  & judge-wrong   (judge slip):      {D}")
if B + C:
    print(f"\n  Of {B + C} EM-misses, the judge rules {B} actually correct (wording) and {C} genuinely wrong.")
    print(f"  => ~{100 * B / (B + C):.0f}% of 'failures' look like metric artifacts, not real errors.")

import json as _json
_json.dump({"judge": JUDGE_KEY, "n": n, "em_acc": A + D, "judge_acc": A + B,
            "clean_correct": A, "wording_artifact": B, "genuine_error": C, "judge_slip": D},
           open("results_judge.json", "w"), indent=1)
print("  saved results_judge.json")

In [ ]:
# ── Few-shot curve: EM/F1 vs number of in-context examples (Qwen-7B) ─────────
# Exemplars are FIXED and NOT from the eval set (no contamination).
import json, torch
_FS_POOL = [
    ("What is the chemical symbol for gold?", "Au"),
    ("Who wrote the play Romeo and Juliet?", "William Shakespeare"),
    ("In which country is the Eiffel Tower located?", "France"),
    ("What is the largest planet in the Solar System?", "Jupiter"),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci"),
]
_fs_key = globals().get("PIPELINE_MODEL_KEY") or next(iter(loaded_models))
_fs_tok, _fs_mdl = loaded_models[_fs_key]

def answer_kshot(question, k):
    shots = "".join(f"Q: {q}\nA: {a}\n\n" for q, a in _FS_POOL[:k])
    user = f"{shots}Q: {question}\nA:"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    try:
        text = _fs_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    except Exception:
        text = f"{SYSTEM_PROMPT}\n\n{user}"
    inp = _fs_tok(text, return_tensors="pt").to(_model_device(_fs_mdl))
    with torch.no_grad():
        out = _fs_mdl.generate(**inp, max_new_tokens=32, do_sample=False, pad_token_id=_fs_tok.eos_token_id)
    return clean_short_answer(_fs_tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True))

_fs_sample = trivia_ds.select(range(20))
_fs_curve = {}
print(f"Few-shot curve on {len(_fs_sample)} TriviaQA examples ({_fs_key}):")
print(f"  {'shots':<8}{'EM':<10}{'F1'}")
for k in [0, 1, 3, 5]:
    em = f1 = 0.0
    for ex in _fs_sample:
        refs = get_answers(ex)
        pred = answer_kshot(ex["question"], k)
        em += exact_match(pred, refs)
        f1 += squad_f1(pred, refs)
    n = len(_fs_sample)
    _fs_curve[k] = {"em": int(em), "n": n, "f1": round(f1 / n, 4)}
    print(f"  {k}-shot   {int(em)}/{n:<7}{f1 / n:.2f}")

json.dump(_fs_curve, open("results_fewshot_curve.json", "w"), indent=1)
print("  saved results_fewshot_curve.json")

# Using SpeechT5 for converting text-to-speech

Motivated by the success of T5 (Text-To-Text Transfer Transformer) in different natural language processing tasks, the unified-modal SpeechT5 framework explores encoder-decoder pre-training for self-supervised speech/text representation learning.

The model is again conveniently available through the HuggingFace Transformers library. The following example illustrates the use of the SpeechT5 model for generating a spectrogram from a textual input, together with a neural vocoder model for producing a speech signal.

More detailed information about SpeechT5 is available on a [tutorial on the HuggingFace blog](https://huggingface.co/blog/speecht5).

In [ ]:
from transformers import AutoProcessor
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, set_seed
from IPython.display import Audio
from datasets import load_dataset
import soundfile as sf
import librosa
import torch

# make results deterministic
set_seed(42)

model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
processor = AutoProcessor.from_pretrained("microsoft/speecht5_tts")

inputs = processor(text="Hello, my dog is cute.", return_tensors="pt")
speaker_embeddings = torch.zeros((1, 512))

# When using SpeechT5 for TTS, you should use "xvector speaker embeddings"
# to customize the output to a particular speaker’s voice characteristics
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[42]["xvector"]).unsqueeze(0)

spectrogram = model.generate_speech(inputs["input_ids"], speaker_embeddings)
with torch.no_grad(): speech = vocoder(spectrogram)

# You can hear the audio inputs
display(Audio(speech.numpy(), rate=16000))

# You can plot the generated spectrogram
import matplotlib.pyplot as plt
plt.figure()
plt.imshow(spectrogram.T)
plt.show()

# You can plot the generated waveform
librosa.display.waveshow(speech.numpy(), sr=16000)

# You can save the audio to a .wav file
sf.write("tts_example.wav", speech.numpy(), samplerate=16000)

## Intermediate tasks:

* Connect the results from your answer to the previous intermediate task (i.e., conditioned language generation) to the SpeechT5 text-to-speech model, so as to produce speech outputs from the text generated by the model. You can also experiment with the use of other text-to-speech models (e.g., [Bark](https://huggingface.co/suno/bark-small), [CSM](https://huggingface.co/sesame/csm-1b), or [MMS-TTS](https://huggingface.co/facebook/mms-tts-eng)).
* Produce naturally-sounding speech-based answers for the first 5 questions in the validation split from the QA dataset used in the previous exercise.
* Connect also the results from your answer to the first intermediate task (i.e., automated speech recognition) to the SpeechT5 model and the LLM, so as to take spoken questions as input and produce a speech output.
* Take the audio samples from 10 TriviaQA questions (as available in connection to the [SLUE-SQA-5 dataset](https://huggingface.co/datasets/asapp/slue-phase-2), in Huggingface datasets), and evaluate the answers generated for the spoken questions using the TER metric.
* Collect audio samples, with your own voice, for the first 2 questions in the validation split from the TriviaQA dataset, and produce naturally-sounding speech-based answers for these two questions.

Notice that ground-truth answers in TriviaQA correspond to relaively short phrases, e.g. directly answering questions through entity names. Thus, evaluations against this dataset based on TER promote models that also produce short answers. However, for conversational systems, it is often preferable to generate longer answers that sound more natural. Hence, in this intermediate step, you should consider adapting the answer generation strategy in order to **produce naturally-sounding answers**.


### Tasks 1 & 2 — Connecting LLM Answers to Text-to-Speech

**Task 1** loads four TTS models and defines a `synthesize_*()` function for each:

| Model | Type | Sample rate |
|---|---|---|
| **SpeechT5** | Transformer encoder-decoder + HifiGan vocoder | 16 kHz |
| **Bark-small** | Generative audio language model | 24 kHz |
| **MMS-TTS** | Lightweight VITS model | ~22 kHz |
| **CSM-1B** | Sesame conversational speech model | 16 kHz |

**Task 2** passes the first 5 TriviaQA validation questions through Qwen3.5-0.8B and synthesizes each answer with all four models, so their voices and quality can be compared side-by-side.

In [ ]:
# ── Free baseline LLMs before the TTS / pipeline sections ───────────────────
# Keeps only the pipeline model in VRAM so a 12B LLM + 4-bit VibeVoice fit a T4.
try:
    free_small_models()
except Exception as _e:
    print('free_small_models unavailable:', _e)

In [ ]:
# ── Task 1 · TTS helpers: SpeechT5, Bark-small, MMS-TTS, CSM-1B ─────────────
import sys, torch, numpy as np
from IPython.display import Audio, display

# ── SpeechT5 (already loaded in demo cell; fall back to reload) ──────────────
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan
from transformers import AutoProcessor as _SPProc

try:
    assert isinstance(model, SpeechT5ForTextToSpeech)
    tts_model, tts_vocoder, tts_proc = model, vocoder, processor
    tts_spk = speaker_embeddings
except (NameError, AssertionError):
    tts_model   = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts')
    tts_vocoder = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan')
    tts_proc    = _SPProc.from_pretrained('microsoft/speecht5_tts')
    from datasets import load_dataset as _ld
    _emb_ds = _ld('Matthijs/cmu-arctic-xvectors', split='validation')
    tts_spk  = torch.tensor(_emb_ds[42]['xvector']).unsqueeze(0)

def synthesize_speecht5(text: str):
    text = text.strip() or 'I do not know.'
    inputs = tts_proc(text=text[:500], return_tensors='pt')
    spectrogram = tts_model.generate_speech(inputs['input_ids'], tts_spk)
    with torch.no_grad():
        speech = tts_vocoder(spectrogram)
    return speech.numpy(), 16000

print('SpeechT5 ready.')

# ── Bark-small (generative; output at 24 kHz) ─────────────────────────────────
from transformers import AutoProcessor as _BarkProc, BarkModel
print('Loading Bark-small...', end=' ', flush=True)
bark_proc  = _BarkProc.from_pretrained('suno/bark-small')
bark_model = BarkModel.from_pretrained('suno/bark-small')
bark_model.eval()
print('done')

def synthesize_bark(text: str):
    text = text.strip() or 'I do not know.'
    inputs = bark_proc(text[:300], voice_preset='v2/en_speaker_6', return_tensors='pt')
    with torch.no_grad():
        audio_arr = bark_model.generate(**inputs, do_sample=True).cpu().numpy().squeeze()
    return audio_arr.astype(np.float32), 24000

# ── MMS-TTS (VITS-based; sample rate from model config) ──────────────────────
from transformers import VitsModel, VitsTokenizer
print('Loading MMS-TTS...', end=' ', flush=True)
mms_tok   = VitsTokenizer.from_pretrained('facebook/mms-tts-eng')
mms_model = VitsModel.from_pretrained('facebook/mms-tts-eng')
mms_model.eval()
print('done')

def synthesize_mms(text: str):
    text = text.strip() or 'I do not know.'
    inputs = mms_tok(text[:500], return_tensors='pt')
    with torch.no_grad():
        output = mms_model(**inputs).waveform
    return output[0].numpy().astype(np.float32), mms_model.config.sampling_rate

# ── CSM-1B (Sesame; conversational TTS, output at 16 kHz) ────────────────────
# Note: 1B params — loading is slow; skip if memory is tight.
from transformers import CsmForConditionalGeneration, AutoProcessor as _CsmProc
csm_model, csm_proc = None, None
print('Loading CSM-1B...', end=' ', flush=True)
try:
    csm_proc  = _CsmProc.from_pretrained('sesame/csm-1b')
    csm_model = CsmForConditionalGeneration.from_pretrained('sesame/csm-1b', torch_dtype=torch.float32)
    csm_model.eval()
    print('done')
except Exception as e:
    print(f'skipped ({e})')

def synthesize_csm(text: str):
    if csm_model is None:
        return np.zeros(16000, dtype=np.float32), 16000
    text = text.strip() or 'I do not know.'
    conversation = [{'role': '0', 'content': [{'type': 'text', 'text': text[:500]}]}]
    inputs = csm_proc.apply_chat_template(
        conversation, tokenize=True, return_dict=True, return_tensors='pt'
    )
    with torch.no_grad():
        audio = csm_model.generate(**inputs, output_audio=True)
    return audio[0].cpu().numpy().astype(np.float32), csm_model.config.sampling_rate

# VibeVoice — modern TTS, loaded 4-bit (bitsandbytes NF4) so it fits alongside a 12B
# LLM on a 15 GB T4: ~2-3 GB instead of ~7 GB in bf16 (needs bitsandbytes >= 0.48.1).
# Lazy-installs the community package and uses its real inference API. Fail-soft: if
# install/load/VRAM fails it falls back to bf16, then the demo loop just skips it.
_vibevoice = {'gen': None, 'tried': False}

def synthesize_vibevoice(text: str):
    text = text.strip() or 'I do not know.'
    if _vibevoice['gen'] is None and not _vibevoice['tried']:
        _vibevoice['tried'] = True
        import subprocess, sys, os
        if not os.path.isdir('VibeVoice'):
            subprocess.run(['git', 'clone', '--depth', '1',
                            'https://github.com/vibevoice-community/VibeVoice.git'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'VibeVoice'], check=True)
        from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
        from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
        from transformers import BitsAndBytesConfig as _BBC
        mid = 'microsoft/VibeVoice-1.5B'      # community mirror: 'vibevoice/VibeVoice-1.5B'
        proc = VibeVoiceProcessor.from_pretrained(mid)
        mdl = None
        if torch.cuda.is_available():
            try:                               # 4-bit quantized (the small-VRAM path)
                _bnb = _BBC(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                            bnb_4bit_compute_dtype=torch.bfloat16)
                mdl = VibeVoiceForConditionalGenerationInference.from_pretrained(
                    mid, quantization_config=_bnb, device_map='auto')
            except Exception as _e:
                print(f'  [VibeVoice] 4-bit load failed ({str(_e)[:50]}); trying bf16')
                mdl = None
        if mdl is None:                        # fallback: bf16 (GPU if available)
            mdl = VibeVoiceForConditionalGenerationInference.from_pretrained(
                mid, torch_dtype=torch.bfloat16,
                device_map='auto' if torch.cuda.is_available() else None)
        mdl.eval()
        try:
            mdl.set_ddpm_inference_steps(num_steps=10)
        except Exception:
            pass
        _vibevoice['gen'] = (proc, mdl)
    if _vibevoice['gen'] is None:
        raise RuntimeError('VibeVoice unavailable')
    proc, mdl = _vibevoice['gen']
    inputs = proc(text=text[:300], speaker_names=['Alice'], return_tensors='pt')
    _dev = next(mdl.parameters()).device
    inputs = {k: (v.to(_dev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(**inputs, cfg_scale=1.3)
    audio = out.audio.squeeze().detach().to(torch.float32).cpu().numpy()
    return audio, 24000

# backward-compat alias used by spoken_qa() in Task 3
synthesize = lambda text: synthesize_speecht5(text)[0]

SYNTHESIZERS = {
    'SpeechT5':  synthesize_speecht5,
    'Bark-small': synthesize_bark,
    'MMS-TTS':   synthesize_mms,
    'CSM-1B':    synthesize_csm,
    'VibeVoice': synthesize_vibevoice,
}
print('All TTS helpers ready.')

# ── Task 2 · Spoken answers for first 5 TriviaQA questions (all TTS models) ──
_pk = globals().get("PIPELINE_MODEL_KEY") or ("Qwen3.5-0.8B" if "Qwen3.5-0.8B" in loaded_models else next(iter(loaded_models)))
tok_qa, mdl_qa = loaded_models[_pk]

print('\n=== Task 2: Spoken Answers for 5 TriviaQA Questions ===\n')
for i, ex in enumerate(trivia_ds.select(range(5))):
    q   = ex['question']
    ans = answer_hf(q, tok_qa, mdl_qa)
    print(f'Q{i+1}: {q}')
    print(f'A{i+1}: {ans}')
    for name, syn_fn in SYNTHESIZERS.items():
        try:
            audio_arr, sr = syn_fn(ans)
            print(f'  [{name}]')
            display(Audio(audio_arr, rate=sr))
        except Exception as e:
            print(f'  [{name}] ERROR: {e}')
    print()


### Task 3 — Full ASR → LLM → TTS Pipeline

Here the three components built throughout the lab are chained into a single `spoken_qa()` function:

1. **ASR** (Whisper-small) — transcribes the input audio to text
2. **LLM** (Qwen3.5-0.8B) — generates a short factual answer
3. **TTS** (SpeechT5) — converts the text answer back to speech

The function is tested with `audio_1` recorded in the ASR section above, producing a complete spoken question-and-answer exchange.

In [ ]:
# -- Task 3 - Full ASR -> LLM -> TTS pipeline
from transformers import AutoProcessor as _WAP, WhisperForConditionalGeneration as _WAM
import torch, numpy as np, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

print("Loading Whisper for ASR...", end=" ", flush=True)
_asr_proc = _WAP.from_pretrained("openai/whisper-small")
_asr_mdl  = _WAM.from_pretrained("openai/whisper-small")
_asr_mdl.eval()
print("done")

def spoken_qa(audio_array: np.ndarray, sample_rate: int = 16000):
    """End-to-end pipeline: audio waveform -> transcription -> LLM answer -> speech."""
    # Step 1: ASR
    inp = _asr_proc(audio=audio_array, sampling_rate=sample_rate, return_tensors="pt")
    with torch.no_grad():
        ids = _asr_mdl.generate(**inp, language="en", task="transcribe")
    question = _asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

    # Step 2: LLM
    _pk = globals().get("PIPELINE_MODEL_KEY") or ("Qwen3.5-0.8B" if "Qwen3.5-0.8B" in loaded_models else next(iter(loaded_models)))
    tok, mdl = loaded_models[_pk]
    answer = answer_hf(question, tok, mdl)

    # Step 3: TTS
    speech = synthesize(answer)

    return question, answer, speech

# Demo: try audio_1 from memory, then fall back to saved recordings on disk
print("\n=== Task 3: Pipeline Demo ===")
_demo_audio = None
_demo_sr    = 16000
_demo_label = ""
try:
    _demo_audio = audio_1
    _demo_label = "audio_1 (from ASR section)"
except NameError:
    for _fname in ("recording_1.wav", "recording_2.wav"):
        _p = Path(".") / _fname
        if _p.exists():
            _arr, _demo_sr = sf.read(str(_p))
            _demo_audio = _arr.astype(np.float32)
            _demo_label = _fname
            break

if _demo_audio is not None:
    print(f"Using: {_demo_label}")
    q_demo, a_demo, sp_demo = spoken_qa(_demo_audio, sample_rate=_demo_sr)
    print(f"Transcribed: {q_demo}")
    print(f"Answer:      {a_demo}")
    display(Audio(sp_demo, rate=16000))
else:
    print("No demo audio found. Record something in the ASR section first.")


### Task 4 — Evaluation on SLUE-SQA-5 Spoken Questions

The [SLUE-SQA-5 dataset](https://huggingface.co/datasets/asapp/slue-phase-2) contains real audio recordings of TriviaQA questions spoken by human speakers. Ten examples are taken from the test split and processed end-to-end through `spoken_qa()`. The generated text answers are then evaluated against the ground-truth answers using the **TER** (Translation Edit Rate) metric — a lower score means fewer edits are needed to match the reference, indicating better answer quality.

In [ ]:
# -- Task 4 - SLUE-SQA-5: spoken QA evaluation with TER
import itertools, numpy as np
from datasets import load_dataset as _ld4
from evaluate import load as _lm4
from IPython.display import Audio, display

N_SLUE = 10
print(f"Loading SLUE-SQA-5 (first {N_SLUE} examples)...")

try:
    slue_raw    = _ld4("asapp/slue-phase-2", "sqa5", split="test",
                        streaming=True, trust_remote_code=True)
    slue_sample = list(itertools.islice(slue_raw, N_SLUE))
    print("Columns:", list(slue_sample[0].keys()))
    loaded_slue = True
except Exception as err:
    print(f"Could not load SLUE dataset: {err}")
    slue_sample = []
    loaded_slue = False

def _normalise_word2time(w2t):
    """Convert word2time to dict mapping word -> list of (start, end) pairs.

    HuggingFace streaming may return word2time in two formats:
      True word dict  : {"the": [0.1, 0.4], "cat": [0.5, 0.8]}
      Columnar dict   : {"word": ["the","cat"], "start": [0.1,0.5],
                          "end": [0.4,0.8]}  (or start_second/end_second)
    """
    if not w2t:
        return {}
    # Columnar detection: has a "word" key whose value is a list of strings
    if "word" in w2t and isinstance(w2t.get("word"), list):
        words  = w2t["word"]
        starts = w2t.get("start_second", w2t.get("start", []))
        ends   = w2t.get("end_second",   w2t.get("end",   []))
        result = {}
        for i, wd in enumerate(words):
            s = float(starts[i]) if i < len(starts) else 0.0
            e = float(ends[i])   if i < len(ends)   else s
            result.setdefault(wd, []).append((s, e))
        return result
    # True word dict
    return w2t

def _word_intervals(times):
    """Normalise word2time values into a list of (start, end) float pairs.

    Handles two storage shapes:
      flat pair    : [0.5, 1.2]           -> [(0.5, 1.2)]
      list-of-pairs: [[0.5, 1.2], ...]    -> [(0.5, 1.2), ...]
      pre-parsed   : [(0.5, 1.2), ...]    -> returned as-is
    """
    if not times:
        return []
    first = times[0]
    # Already a tuple (from _normalise_word2time)
    if isinstance(first, tuple):
        return list(times)
    # Flat pair: [float, float]
    if isinstance(first, (int, float)):
        return [(float(times[0]), float(times[1]))]
    # List-of-pairs: [[float, float], ...]
    return [(float(iv[0]), float(iv[1])) for iv in times]

def _slue_ref_text(ex):
    """Extract reference answer text from a SLUE-SQA-5 example."""
    spans     = ex.get("answer_spans") or {}
    word2time = _normalise_word2time(ex.get("word2time") or {})
    doc_text  = (ex.get("normalized_document_text")
                 or ex.get("raw_document_text") or "")

    if not spans:
        return ""

    # Columnar dict: {"start_second": [...], "end_second": [...], ...}
    if isinstance(spans, dict):
        if spans.get("text") and spans["text"]:
            return spans["text"][0]
        starts = spans.get("start_second", spans.get("start", []))
        ends   = spans.get("end_second",   spans.get("end",   []))
        if not starts:
            return ""
        t0 = float(starts[0])
        t1 = float(ends[0]) if ends else t0
    # Row-format list: [{"start_second": 2.5, "end_second": 4.0}]
    elif isinstance(spans, list) and spans:
        span = spans[0]
        if isinstance(span, str):
            return span
        if span.get("text"):
            return span["text"]
        t0 = float(span.get("start_second", span.get("start", 0)))
        t1 = float(span.get("end_second",   span.get("end",   t0)))
    else:
        return ""

    # Collect words whose time interval falls within [t0, t1]
    answer_words = []
    for word, times in word2time.items():
        for ws, we in _word_intervals(times):
            if ws >= t0 - 0.05 and we <= t1 + 0.05:
                answer_words.append((ws, word))
                break
    if answer_words:
        answer_words.sort()
        return " ".join(w for _, w in answer_words)

    # Last resort: char-based slice
    c0 = spans.get("start_char", -1) if isinstance(spans, dict) else -1
    c1 = spans.get("end_char",   -1) if isinstance(spans, dict) else -1
    if doc_text and c0 >= 0 and c1 > c0:
        return doc_text[c0:c1]
    return ""

if loaded_slue and slue_sample:
    slue_preds, slue_refs = [], []

    for i, ex in enumerate(slue_sample):
        q_audio_field = ex.get("question_audio") or {}
        try:
            audio_arr = np.array(q_audio_field["array"]).astype(np.float32)
            sr        = q_audio_field["sampling_rate"]
            q_text, a_text, a_speech = spoken_qa(audio_arr, sample_rate=sr)
        except Exception:
            q_text   = (ex.get("normalized_question_text")
                        or ex.get("raw_question_text", ""))
            tok_s, mdl_s = loaded_models.get("Qwen3.5-0.8B") or next(iter(loaded_models.values()))
            a_text   = answer_hf(q_text, tok_s, mdl_s)
            a_speech = synthesize(a_text)

        ref = _slue_ref_text(ex)

        slue_preds.append(a_text)
        slue_refs.append([ref] if ref else [" "])
        print(f"[{i+1}] Q:   {q_text[:65]}")
        print(f"      A:   {a_text}")
        print(f"      Ref: {ref}")
        display(Audio(a_speech, rate=16000))

    # Clean + normalize before scoring — verbose answers vs short spans otherwise
    # give TER > 100%. Skip examples whose extracted reference span is empty.
    clean_preds = [clean_short_answer(p) for p in slue_preds]
    flat_refs   = [r[0] if r else "" for r in slue_refs]
    valid = [(p, r) for p, r in zip(clean_preds, flat_refs) if normalize_answer(r)]

    em = sum(exact_match(p, [r]) for p, r in valid)
    f1 = (sum(squad_f1(p, [r]) for p, r in valid) / len(valid)) if valid else 0.0

    ter_metric = _lm4("ter")
    ter_hyp = [normalize_answer(p) or "<blank>" for p, r in valid]
    ter_ref = [[normalize_answer(r)] for p, r in valid]
    slue_ter = ter_metric.compute(predictions=ter_hyp, references=ter_ref) if valid else {"score": float("nan")}

    print(f"\nResults on {len(valid)}/{N_SLUE} scored SLUE-SQA-5 examples")
    print(f"  Exact-match: {em}/{len(valid)}")
    print(f"  Token-F1:    {f1:.3f}")
    print(f"  TER:         {slue_ter['score']:.2f}  (lower is better)")


### Task 5 — Spoken Answers for Your Own Voice Recordings

The first two TriviaQA validation questions are shown on screen. You read each question aloud and the recording is saved to `trivia_question_1.wav` / `trivia_question_2.wav`. Each audio clip is then run through the full `spoken_qa()` pipeline: Whisper transcribes your question, Qwen3.5-0.8B answers it, and SpeechT5 speaks the answer back.

Set `FORCE_TQ1 = True` or `FORCE_TQ2 = True` to record a new take; leave them `False` to reuse a saved recording.

In [ ]:
# -- Task 5 - Record yourself asking TriviaQA question 1
import soundfile as sf, numpy as np
from pathlib import Path
from IPython.display import Audio, display

# fallback: redefine _do_record if ASR recording cell was not run
try:
    _do_record
except NameError:
    import sounddevice as _sd
    def _do_record(duration=5):
        print(f"  Recording {duration}s -- speak now!")
        _audio = _sd.rec(int(duration * 16000), samplerate=16000, channels=1, dtype="float32")
        _sd.wait()
        print("  Done.")
        return _audio.squeeze()

TQ1_PATH  = Path(".") / "trivia_question_1.wav"
FORCE_TQ1 = False # set True to re-record

print("Say this question aloud:")
print(f'  "{trivia_ds[0]["question"]}"\n')

if TQ1_PATH.exists() and not FORCE_TQ1:
    print(f"Loaded existing {TQ1_PATH.name}  (set FORCE_TQ1=True to re-record)")
    audio_tq1, _ = sf.read(str(TQ1_PATH))
    audio_tq1    = audio_tq1.astype(np.float32)
else:
    print(f"Recording -> {TQ1_PATH.name}")
    audio_tq1 = _do_record(duration=6)
    sf.write(str(TQ1_PATH), audio_tq1, 16000)
    print(f"Saved to {TQ1_PATH}")

print("Your recording:")
display(Audio(audio_tq1, rate=16000))

q1, a1, sp1 = spoken_qa(audio_tq1)
print(f"Transcribed:  {q1}")
print(f"Answer:       {a1}")
print("Spoken answer:")
display(Audio(sp1, rate=16000))


In [ ]:
# -- Task 5 - Record yourself asking TriviaQA question 2
import soundfile as sf, numpy as np
from pathlib import Path
from IPython.display import Audio, display

# fallback: redefine _do_record if ASR recording cell was not run
try:
    _do_record
except NameError:
    import sounddevice as _sd
    def _do_record(duration=5):
        print(f"  Recording {duration}s -- speak now!")
        _audio = _sd.rec(int(duration * 16000), samplerate=16000, channels=1, dtype="float32")
        _sd.wait()
        print("  Done.")
        return _audio.squeeze()

TQ2_PATH  = Path(".") / "trivia_question_2.wav"
FORCE_TQ2 = False  # set True to re-record

print("Say this question aloud:")
print(f'  "{trivia_ds[1]["question"]}"\n')

if TQ2_PATH.exists() and not FORCE_TQ2:
    print(f"Loaded existing {TQ2_PATH.name}  (set FORCE_TQ2=True to re-record)")
    audio_tq2, _ = sf.read(str(TQ2_PATH))
    audio_tq2    = audio_tq2.astype(np.float32)
else:
    print(f"Recording -> {TQ2_PATH.name}")
    audio_tq2 = _do_record(duration=6)
    sf.write(str(TQ2_PATH), audio_tq2, 16000)
    print(f"Saved to {TQ2_PATH}")

print("Your recording:")
display(Audio(audio_tq2, rate=16000))

q2, a2, sp2 = spoken_qa(audio_tq2)
print(f"Transcribed:  {q2}")
print(f"Answer:       {a2}")
print("Spoken answer:")
display(Audio(sp2, rate=16000))


In [ ]:
# ── TTS intelligibility: round-trip WER (text → TTS → Whisper → WER) ─────────
# "ASR+TTS in one bucket": synthesize known sentences, transcribe them back, measure WER.
# Reported through BOTH whisper-large-v3 (~0% on clean speech → isolates TTS quality) and
# whisper-small (the pipeline's ASR → realistic combined ASR+TTS). This measures
# INTELLIGIBILITY (are the words recoverable), NOT naturalness. Fail-soft per synthesizer.
import string, gc, json, torch, librosa
from evaluate import load as _wl
from transformers import AutoProcessor as _RAP, WhisperForConditionalGeneration as _RWM

_wer_rt = _wl("wer")
RT_SENTENCES = [
    "Who is the chief executive officer of the company?",
    "The highest mountain in Africa is Kilimanjaro.",
    "The Japanese share index is called the Nikkei.",
    "Rita Coolidge sang the title song for Octopussy.",
    "The capital of Niger is Niamey.",
    "Diana Ross had a number one single called Upside Down.",
]

def _wnorm(s):
    return " ".join(s.lower().translate(str.maketrans("", "", string.punctuation)).split())

# whisper-small: reuse the pipeline ASR if already loaded, else load it
try:
    _sm_p, _sm_m = _asr_proc, _asr_mdl
except NameError:
    _sm_p = _RAP.from_pretrained("openai/whisper-small")
    _sm_m = _RWM.from_pretrained("openai/whisper-small"); _sm_m.eval()

# whisper-large-v3 as the high-quality "ear" (GPU fp16; freed at the end)
print("Loading whisper-large-v3 ...", end=" ", flush=True)
_lp = _RAP.from_pretrained("openai/whisper-large-v3")
_lm = _RWM.from_pretrained("openai/whisper-large-v3",
                           torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
_lm = _lm.to("cuda") if torch.cuda.is_available() else _lm
_lm.eval(); print("done")

print(f"\n{'TTS model':<14}{'WER large-v3':>14}{'WER small':>12}   (lower = more intelligible)")
print("─" * 54)
rt_results = {}
for name, synth in SYNTHESIZERS.items():
    try:
        refs, hyp_l, hyp_s = [], [], []
        for t in RT_SENTENCES:
            audio, sr = synth(t)
            audio = audio.astype("float32")
            a16 = librosa.resample(audio, orig_sr=sr, target_sr=16000) if sr != 16000 else audio
            refs.append(_wnorm(t))
            hyp_l.append(_wnorm(transcribe(a16, language="en", proc=_lp, mdl=_lm)))
            hyp_s.append(_wnorm(transcribe(a16, language="en", proc=_sm_p, mdl=_sm_m)))
        wl = _wer_rt.compute(predictions=hyp_l, references=refs)
        ws = _wer_rt.compute(predictions=hyp_s, references=refs)
        rt_results[name] = {"wer_large_v3": round(wl, 4), "wer_small": round(ws, 4)}
        print(f"{name:<14}{wl * 100:>13.1f}%{ws * 100:>11.1f}%")
    except Exception as e:
        print(f"{name:<14}  skipped: {str(e)[:34]}")

del _lm, _lp
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

json.dump(rt_results, open("tts_roundtrip_wer.json", "w"), indent=1)
print("\nRound-trip WER = ASR+TTS in one bucket: large-v3 isolates TTS, small = realistic pipeline.")
print("Measures intelligibility (words recoverable), not naturalness. Saved tts_roundtrip_wer.json")

### (Bonus) MisoTTS — emotive 8B TTS

MisoTTS (`MisoLabs/MisoTTS`, ~8B, released June 2026) is too large for a Colab T4 and has no published benchmarks yet, so we treat it as an **experimental** comparison: samples are rendered offline on the RTX 5090 with `gen_misotts_samples.py` and loaded below.

In [ ]:
# ── (Bonus) MisoTTS samples — rendered offline on the RTX 5090 ───────────────
# MisoTTS (~8B, float32 ≈ 20 GB) does not fit a Colab T4, so a few answer clips are
# rendered offline with gen_misotts_samples.py and loaded here for comparison.
from pathlib import Path
from IPython.display import Audio, display
import soundfile as sf

_miso_dir = Path("misotts_samples")
_wavs = sorted(_miso_dir.glob("*.wav")) if _miso_dir.is_dir() else []
if _wavs:
    print(f"MisoTTS bonus samples ({len(_wavs)}):")
    for w in _wavs:
        arr, sr = sf.read(str(w))
        print(f"  {w.name}")
        display(Audio(arr, rate=sr))
else:
    print("No misotts_samples/ found.")
    print("Run gen_misotts_samples.py on a GPU box (e.g. the RTX 5090) and copy the folder here.")

# Main problem

Students are tasked with joining together the speech recognition/translation, language understanding and generation, and text-to-speech components, in order to build a turn-based conversational spoken question answering approach.

* The method should take as input speech utterances with questions, e.g.loading the audio file(s) that simulate, in an off-line experiment supported by the Python notebook, questions given in multipe turns.
* The language understanding and generation component should use as input a transcription/translation for each speech utterance, and optionally also transcriptions/translations for the previous speech utterances (i.e., the different turns in the same conversation context).
* The language understanding and generation component can explore different strategies for improving answer quality:
  * Use of LLMs trained to follow instructions or capable of performing reasoning, e.g. with reinforcement learning from human feedback.
  * Prompting the language model with retrieved in-context examples.
  * Using parameter-efficient fine-ting with existing conversational question answering datasets (e.g., [the CoQA dataset](https://stanfordnlp.github.io/coqa/), which is [also available](https://huggingface.co/datasets/stanfordnlp/coqa) from HuggingFace datasets).
  * ...
* The text-to-speech component should take as input the results from language generation, and produce a speech output for each question.
* To evaluate the proposed method in an off-line experiment, students must collect small audio samples, with their own voices, for the different questions in one of the instances in the CoQA validation split. Having these files stored in a given directory, the notebook should show the turn-based results produced for the different questions.


Notice that the automated speech recognition, language understanding/generation, and the text-to-speech components, all can explore different approaches from the main suggestions in the previous exercises, although students should attempt to justify their choices (e.g., if changing the automated speech recognition component, show that the alternative achieves a lower WER).

Students can also attempt to further streamline/aggregate the overall approach, e.g. using common models to perform speech recognition, language generation, and text-to-speech (e.g., using models such as [Phi-4-multimodal-instruct](https://huggingface.co/microsoft/Phi-4-multimodal-instruct), which can address the multiple tasks that are involved, or instead using full-duplex speech communication models like [Moshi](https://github.com/kyutai-labs/moshi) or [Moshi-RAG](https://github.com/kyutai-labs/moshi-rag) to would allow one to go beyond turn-based interactions).

## Main Problem — Turn-Based Spoken Conversational QA

This section assembles the ASR, LLM, and TTS components into a full **multi-turn spoken dialogue system** evaluated on the [CoQA dataset](https://huggingface.co/datasets/stanfordnlp/coqa).

Each CoQA instance provides a **story passage** and a sequence of questions whose answers depend on both the passage and the preceding turns — making context tracking essential.

The pipeline per turn:
1. Load the recorded audio for that question
2. **ASR** (Whisper) — transcribe the spoken question
3. **LLM** (Qwen3.5-0.8B) — answer using the passage + full conversation history
4. **TTS** (SpeechT5) — synthesize the answer as speech
5. Append *(question, answer)* to the running history for the next turn

Finally, **TER** is computed over all generated answers vs. the CoQA ground truth.

In [ ]:
# ── Main Problem · Load CoQA and inspect the chosen story ───────────────────
import sys
!{sys.executable} -m pip install sacrebleu -q

from datasets import load_dataset

coqa_ds  = load_dataset('stanfordnlp/coqa', split='validation')

STORY_IDX = 0  # change to try a different CoQA story

story      = coqa_ds[STORY_IDX]
passage    = story['story']
questions  = story['questions']
gt_answers = story['answers']['input_text']

print(f'Source: {story["source"]}  |  Questions: {len(questions)}')
print(f'\nPassage:\n{passage[:600]}...\n')
print('Questions and ground-truth answers:')
for i, (q, a) in enumerate(zip(questions, gt_answers)):
    print(f'  Q{i+1}: {q}')
    print(f'  A{i+1}: {a}')
    print()


In [ ]:
# ── Main Problem · Record yourself asking each CoQA question ────────────────
import soundfile as sf, numpy as np
from pathlib import Path
from IPython.display import Audio, display

COQA_DIR = Path('.') / 'coqa_recordings'
COQA_DIR.mkdir(exist_ok=True)

# One entry per question — set to True to re-record that slot
FORCE_RECORD = [False] * len(questions)
# Example: to re-record only question 2, set FORCE_RECORD[1] = True

coqa_audios = []
for i, q in enumerate(questions):
    wav_path = COQA_DIR / f'coqa_q{i+1:02d}.wav'
    print(f'Q{i+1}: {q}')
    if wav_path.exists() and not FORCE_RECORD[i]:
        print(f'  Loaded {wav_path.name}  (set FORCE_RECORD[{i}]=True to re-record)')
        audio, _ = sf.read(str(wav_path))
        audio = audio.astype(np.float32)
    else:
        print(f'  Recording -> {wav_path.name}')
        audio = _do_record(duration=6)
        sf.write(str(wav_path), audio, 16000)
        print('  Saved.')
    display(Audio(audio, rate=16000))
    coqa_audios.append(audio)
    print()


In [ ]:
# ── Main Problem · (optional) load the CoQA fine-tuned LoRA adapter ──────────
# Train it offline with finetune_coqa_qlora.py on a GPU box (e.g. the RTX 5090),
# then place the resulting ./coqa_lora folder next to this notebook (or mount it).
# If the adapter is absent, the CoQA pipeline simply uses the base model.
import os

COQA_ADAPTER_DIR = "coqa_lora"
COQA_MODEL_KEY = PIPELINE_MODEL_KEY          # default: base model

if PIPELINE_MODEL_KEY is not None and os.path.isdir(COQA_ADAPTER_DIR):
    try:
        from peft import PeftModel
        _btok, _bmdl = loaded_models[PIPELINE_MODEL_KEY]
        _ft = PeftModel.from_pretrained(_bmdl, COQA_ADAPTER_DIR)
        _ft.eval()
        loaded_models["CoQA-FT"] = (_btok, _ft)
        COQA_MODEL_KEY = "CoQA-FT"
        print(f"Loaded CoQA adapter from ./{COQA_ADAPTER_DIR} -> 'CoQA-FT'")
    except Exception as e:
        print(f"Adapter load failed ({str(e)[:70]}); using base model {PIPELINE_MODEL_KEY}.")
else:
    print(f"No ./{COQA_ADAPTER_DIR} adapter found; using base model {PIPELINE_MODEL_KEY} for CoQA.")

In [ ]:
# ── Main Problem · Multi-turn ASR -> LLM -> TTS pipeline ────────────────────
import torch, numpy as np
from transformers import AutoProcessor as _WAP2, WhisperForConditionalGeneration as _WAM2
from IPython.display import Audio, display

# Reload Whisper if not already in scope from tts-task3
try:
    _asr_proc, _asr_mdl
except NameError:
    print('Loading Whisper...', end=' ', flush=True)
    _asr_proc = _WAP2.from_pretrained('openai/whisper-small')
    _asr_mdl  = _WAM2.from_pretrained('openai/whisper-small')
    _asr_mdl.eval()
    print('done')

# Reload SpeechT5 synthesize() if not in scope from tts-task12
try:
    synthesize
except NameError:
    from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, AutoProcessor as _SP2
    _tts2_model   = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts')
    _tts2_vocoder = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan')
    _tts2_proc    = _SP2.from_pretrained('microsoft/speecht5_tts')
    from datasets import load_dataset as _ld2
    _emb2 = _ld2('Matthijs/cmu-arctic-xvectors', split='validation')
    _spk2 = torch.tensor(_emb2[42]['xvector']).unsqueeze(0)
    def synthesize(text):
        text = text.strip() or 'I do not know.'
        inp = _tts2_proc(text=text[:500], return_tensors='pt')
        spec = _tts2_model.generate_speech(inp['input_ids'], _spk2)
        with torch.no_grad(): sp = _tts2_vocoder(spec)
        return sp.numpy()

CONV_SYSTEM = (
    'You are a helpful conversational QA assistant. '
    'Answer questions based only on the provided passage. '
    'Keep answers brief: one short sentence or a few words only.'
)

def build_coqa_prompt(passage, history, question):
    """Passage + conversation history + current question as a single user message."""
    hist_text = '\n'.join(f'Q: {q}\nA: {a}' for q, a in history)
    body = f'Passage:\n{passage[:1200]}'
    if hist_text:
        body += f'\n\nConversation so far:\n{hist_text}'
    body += f'\n\nCurrent question: {question}\nAnswer:'
    return body

_pk = globals().get("COQA_MODEL_KEY") or globals().get("PIPELINE_MODEL_KEY") or ("Qwen3.5-0.8B" if "Qwen3.5-0.8B" in loaded_models else next(iter(loaded_models)))
tok_conv, mdl_conv = loaded_models[_pk]

history           = []   # list of (transcribed_q, generated_a) tuples
generated_answers = []

print('=== Multi-Turn Spoken QA on CoQA ===\n')
for i, (audio, question) in enumerate(zip(coqa_audios, questions)):
    print(f'--- Turn {i+1} / {len(questions)} ---')

    # 1 · ASR
    inp = _asr_proc(audio=audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        ids = _asr_mdl.generate(**inp, language="en", task="transcribe")
    transcribed_q = _asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
    print(f'Heard:     {transcribed_q}')

    # 2 · LLM with conversation history
    user_msg = build_coqa_prompt(passage, history, transcribed_q)
    messages = [
        {'role': 'system', 'content': CONV_SYSTEM},
        {'role': 'user',   'content': user_msg},
    ]
    try:
        input_text = tok_conv.apply_chat_template(
            messages, enable_thinking=False, add_generation_prompt=True, tokenize=False
        )
    except TypeError:
        input_text = tok_conv.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
    inputs = tok_conv(input_text, return_tensors='pt').to(_model_device(mdl_conv))
    with torch.no_grad():
        out = mdl_conv.generate(
            **inputs, max_new_tokens=80, do_sample=False,
            pad_token_id=tok_conv.eos_token_id
        )
    raw = tok_conv.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    if '</think>' in raw:
        raw = raw.split('</think>')[-1].strip()
    for sep in ('.', '\n'):
        idx = raw.find(sep)
        if 0 < idx < 200:
            raw = raw[:idx + 1]
            break
    answer = raw.strip()
    print(f'Generated: {answer}')
    print(f'GT answer: {gt_answers[i]}')

    # 3 · TTS
    speech = synthesize(answer)
    print('Spoken answer:')
    display(Audio(speech, rate=16000))

    # 4 · Update conversation history
    history.append((transcribed_q, answer))
    generated_answers.append(answer)
    print()


In [ ]:
# ── Main Problem · TER (+ EM / F1) evaluation ───────────────────────────────
from evaluate import load as _lm_main

ter_metric = _lm_main('ter')

# Clean + normalize before scoring — short CoQA references vs verbose generations
# otherwise yield the spurious TER > 100% seen in the first run.
clean_gen = [clean_short_answer(g) for g in generated_answers]
n = len(clean_gen)
em_hits = sum(exact_match(clean_gen[i], [gt_answers[i]]) for i in range(n))
f1_mean = (sum(squad_f1(clean_gen[i], [gt_answers[i]]) for i in range(n)) / n) if n else 0.0

ter_hyp = [normalize_answer(g) or "<blank>" for g in clean_gen]
ter_ref = [[normalize_answer(gt_answers[i]) or "<blank>"] for i in range(n)]
ter_result = ter_metric.compute(predictions=ter_hyp, references=ter_ref)

print(f'=== Evaluation over {n} CoQA turns ===')
print(f'Exact-match: {em_hits}/{n}  ({100 * em_hits / n:.1f}%)' if n else 'no turns')
print(f'Token-F1:    {f1_mean:.3f}')
print(f'TER:         {ter_result["score"]:.2f}  (lower is better)\n')

W = 38
print(f'{"Turn":<6} {"Generated":<{W}} {"Ground Truth":<{W}}')
print('-' * (6 + 2 * W + 2))
for i, (gen, gt) in enumerate(zip(clean_gen, gt_answers)):
    print(f'T{i+1:<5} {gen[:W-1]:<{W}} {gt[:W-1]:<{W}}')

In [ ]:
# ── End-to-end latency: per-component timing of the spoken-QA pipeline ───────
# It's a dialogue system — how fast is each turn? ASR + LLM + TTS timed separately.
# (ASR & TTS run on CPU here, the LLM on GPU — typical for this setup.)
import time, json, statistics, torch
_lat_key = (globals().get("COQA_MODEL_KEY") or globals().get("PIPELINE_MODEL_KEY")
            or next(iter(loaded_models)))
_lat_tok, _lat_mdl = loaded_models[_lat_key]
_lat_audios = (coqa_audios[:4] if "coqa_audios" in globals() and coqa_audios else
               ([audio_1] if "audio_1" in globals() else []))

def _stage_times(aud):
    t0 = time.perf_counter()
    inp = _asr_proc(audio=aud, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        ids = _asr_mdl.generate(**inp, language="en", task="transcribe")
    q = _asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
    t1 = time.perf_counter()
    a = answer_hf(q, _lat_tok, _lat_mdl)
    t2 = time.perf_counter()
    _ = synthesize(a)
    t3 = time.perf_counter()
    return t1 - t0, t2 - t1, t3 - t2

if _lat_audios:
    _stage_times(_lat_audios[0])                      # warm-up (CUDA init / caches)
    asr_t, llm_t, tts_t = [], [], []
    for aud in _lat_audios:
        a_, l_, t_ = _stage_times(aud)
        asr_t.append(a_); llm_t.append(l_); tts_t.append(t_)
    m = statistics.mean
    total = m(asr_t) + m(llm_t) + m(tts_t)
    verdict = "interactive" if total < 2 else "near-interactive" if total < 5 else "batch-only"
    print(f"Per-turn latency over {len(_lat_audios)} turns (ASR+TTS on CPU, LLM={_lat_key} on GPU):")
    print(f"  ASR (Whisper-small): {m(asr_t):6.2f} s")
    print(f"  LLM (answer):        {m(llm_t):6.2f} s")
    print(f"  TTS (SpeechT5):      {m(tts_t):6.2f} s")
    print(f"  ----------------------------")
    print(f"  Total per turn:      {total:6.2f} s   ({verdict})")
    json.dump({"asr_s": round(m(asr_t), 3), "llm_s": round(m(llm_t), 3),
               "tts_s": round(m(tts_t), 3), "total_s": round(total, 3),
               "n_turns": len(_lat_audios), "llm": _lat_key, "verdict": verdict},
              open("results_latency.json", "w"), indent=1)
    print("  saved results_latency.json")
else:
    print("No audio available for latency timing (need coqa_audios or audio_1).")

In [ ]:
# ── Main Problem · base vs CoQA-fine-tuned comparison (text-only, gold questions) ──
# Isolates the LLM effect by using the gold questions (no ASR) and gold history, so
# the EM / F1 / TER difference reflects the fine-tune, not ASR noise.
import contextlib
from evaluate import load as _lm_cmp

_ter = _lm_cmp("ter")


def _coqa_generate(mdl, tok, passage, history, question, max_new_tokens=64):
    msgs = [{"role": "system", "content": CONV_SYSTEM},
            {"role": "user",   "content": build_coqa_prompt(passage, history, question)}]
    try:
        text = tok.apply_chat_template(msgs, enable_thinking=False,
                                       add_generation_prompt=True, tokenize=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = tok(text, return_tensors="pt").to(_model_device(mdl))
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_short_answer(raw)


def _run_variant(mdl, tok, use_adapter):
    preds, history = [], []
    ctx = (mdl.disable_adapter() if (hasattr(mdl, "disable_adapter") and not use_adapter)
           else contextlib.nullcontext())
    with ctx:
        for q, gt in zip(questions, gt_answers):
            preds.append(_coqa_generate(mdl, tok, passage, history, q))
            history.append((q, gt))            # gold history (teacher forcing)
    n = len(preds)
    em = sum(exact_match(preds[i], [gt_answers[i]]) for i in range(n))
    f1 = sum(squad_f1(preds[i], [gt_answers[i]]) for i in range(n)) / max(1, n)
    hyp = [normalize_answer(p) or "<blank>" for p in preds]
    ref = [[normalize_answer(gt_answers[i]) or "<blank>"] for i in range(n)]
    ter = _ter.compute(predictions=hyp, references=ref)["score"]
    return preds, em, f1, ter, n


print("=== CoQA: base vs fine-tuned (gold questions) ===")
if "CoQA-FT" in loaded_models:
    ft_tok, ft_mdl = loaded_models["CoQA-FT"]
    base_preds, b_em, b_f1, b_ter, n = _run_variant(ft_mdl, ft_tok, use_adapter=False)
    ft_preds,   f_em, f_f1, f_ter, _ = _run_variant(ft_mdl, ft_tok, use_adapter=True)
    print(f"{'Variant':<10}{'EM':>9}{'F1':>9}{'TER':>9}")
    print(f"{'base':<10}{str(b_em)+'/'+str(n):>9}{b_f1:>9.3f}{b_ter:>9.2f}")
    print(f"{'CoQA-FT':<10}{str(f_em)+'/'+str(n):>9}{f_f1:>9.3f}{f_ter:>9.2f}")
    print()
    W = 30
    print(f"{'Q#':<4}{'base':<{W}}{'CoQA-FT':<{W}}{'gold':<{W}}")
    for i in range(n):
        print(f"{i+1:<4}{base_preds[i][:W-1]:<{W}}{ft_preds[i][:W-1]:<{W}}{gt_answers[i][:W-1]:<{W}}")
else:
    bt, bm = loaded_models[PIPELINE_MODEL_KEY]
    _, em, f1, ter, n = _run_variant(bm, bt, use_adapter=True)
    print(f"(no adapter loaded — base only)  EM {em}/{n}  F1 {f1:.3f}  TER {ter:.2f}")
    print("Train one with finetune_coqa_qlora.py to populate the comparison.")

In [ ]:
# ── CoQA over multiple stories (gold questions) — robustness beyond n=1 ──────
# The audio pipeline ran ONE story. Here we score the LLM on several stories using gold
# questions, RESETTING conversation memory PER STORY (each story is its own dialogue).
# No ASR/TTS (we only recorded one story) — this isolates dialogue-QA quality on more data.
import json, torch
from evaluate import load as _lm_ms

_ms_ter = _lm_ms("ter")
N_STORIES = 5
_ms_key = (globals().get("COQA_MODEL_KEY") or globals().get("PIPELINE_MODEL_KEY")
           or next(iter(loaded_models)))
_ms_tok, _ms_mdl = loaded_models[_ms_key]

def _coqa_answer(passage, history, question):
    msgs = [{"role": "system", "content": CONV_SYSTEM},
            {"role": "user", "content": build_coqa_prompt(passage, history, question)}]
    try:
        text = _ms_tok.apply_chat_template(msgs, enable_thinking=False,
                                           add_generation_prompt=True, tokenize=False)
    except TypeError:
        text = _ms_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = _ms_tok(text, return_tensors="pt").to(_model_device(_ms_mdl))
    with torch.no_grad():
        out = _ms_mdl.generate(**inp, max_new_tokens=64, do_sample=False, pad_token_id=_ms_tok.eos_token_id)
    return clean_short_answer(_ms_tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True))

all_em = all_f1 = all_n = 0
hyps_all, refs_all, per_story = [], [], []
print(f"CoQA over {N_STORIES} stories (gold questions, memory reset per story), model={_ms_key}:")
print(f"  {'story':<7}{'turns':<7}{'EM':<10}{'F1'}")
for s in range(N_STORIES):
    story = coqa_ds[s]
    passage, qs, golds = story["story"], story["questions"], story["answers"]["input_text"]
    history = []                                    # <-- memory resets per story
    s_em = s_f1 = 0.0
    for q, gt in zip(qs, golds):
        pred = _coqa_answer(passage, history, q)
        s_em += exact_match(pred, [gt]); s_f1 += squad_f1(pred, [gt])
        hyps_all.append(normalize_answer(pred) or "<blank>")
        refs_all.append([normalize_answer(gt) or "<blank>"])
        history.append((q, gt))                     # gold history (teacher forcing)
    nt = len(qs)
    all_em += s_em; all_f1 += s_f1; all_n += nt
    per_story.append({"story": s, "turns": nt, "em": int(s_em), "f1": round(s_f1 / nt, 4)})
    print(f"  {s:<7}{nt:<7}{int(s_em)}/{nt:<7}{s_f1 / nt:.2f}")

ter = _ms_ter.compute(predictions=hyps_all, references=refs_all)["score"]
print(f"  ----------------------------")
print(f"  TOTAL: EM {int(all_em)}/{all_n} ({100 * all_em / all_n:.1f}%)   F1 {all_f1 / all_n:.2f}   TER {ter:.2f}")
json.dump({"n_stories": N_STORIES, "total_turns": all_n, "em": int(all_em),
           "em_pct": round(100 * all_em / all_n, 1), "f1": round(all_f1 / all_n, 4),
           "ter": round(ter, 2), "per_story": per_story},
          open("results_coqa_multistory.json", "w"), indent=1)
print("  saved results_coqa_multistory.json")